# 🏆 Pest Classification Tournament: Finding the Ultimate Prediction Model

## Project Overview

Welcome to our comprehensive **tournament-style classification analysis** designed to identify the best performing model for predicting pest occurrence (`NEW_CATCHES`). This notebook represents the culmination of our data science pipeline, where we pit multiple machine learning algorithms against each other in a rigorous competition.

### Tournament Structure

Our tournament is organized into three distinct phases:

1. **🥊 Standard Classifiers Tournament**: Traditional machine learning algorithms compete
   - RandomForestClassifier
   - XGBoostClassifier 
   - LightGBMClassifier

2. **🧠 Deep Learning Tournament**: Advanced neural networks battle for supremacy
   - LSTM (Long Short-Term Memory)
   - GRU (Gated Recurrent Unit)
   - CNN-LSTM Hybrid

3. **🏅 Grand Finale**: The champions from each category face off to determine the ultimate winner

### Key Principles

- **No Data Leakage**: All splits maintain chronological order (shuffle=False)
- **Robust Imbalance Handling**: Class weighting and optimal threshold tuning
- **Interactive Visualizations**: All plots use Plotly for enhanced exploration
- **Comprehensive Evaluation**: F1-score, AUC, precision, recall, and confusion matrices

Let the tournament begin! 🚀

## 📦 Global Setup and Library Imports

We begin by importing all necessary libraries for our complete analysis. This includes traditional machine learning libraries, deep learning frameworks, visualization tools, and utility packages.

In [13]:
%pip install tensorflow==2.15.0
%pip install scikit-learn pandas numpy plotly
%pip install xgboost lightgbm

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [14]:
# Core Data Science Libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Machine Learning - Core
from sklearn.model_selection import train_test_split, GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, 
    roc_curve, f1_score, precision_recall_curve, accuracy_score
)
from sklearn.utils.class_weight import compute_class_weight

# Machine Learning - Algorithms
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout, Conv1D, MaxPooling1D, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff

# Utilities
import joblib
import json
from datetime import datetime
import os

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("✅ All libraries imported successfully!")
print(f"📊 Pandas version: {pd.__version__}")
print(f"🤖 TensorFlow version: {tf.__version__}")
print(f"📈 Plotly available for interactive visualizations")

✅ All libraries imported successfully!
📊 Pandas version: 2.2.2
🤖 TensorFlow version: 2.15.0
📈 Plotly available for interactive visualizations


## 📁 Data Loading and Initial Exploration

We load both our engineered dataset (for modeling) and the merged dataset (for additional context). The engineered dataset contains all our carefully crafted features, while the merged dataset provides the complete picture of our data transformation journey.

In [15]:
# Load the datasets
print("📖 Loading datasets...")

# Primary dataset for modeling (engineered features)
df_engineered = pd.read_csv('cleaned_engineered_data.csv')

# Secondary dataset for context
df_merged = pd.read_csv('cleaned_merged_data.csv')

print(f"✅ Engineered dataset loaded: {df_engineered.shape[0]:,} rows × {df_engineered.shape[1]} columns")
print(f"✅ Merged dataset loaded: {df_merged.shape[0]:,} rows × {df_merged.shape[1]} columns")

# Display basic information about our primary dataset
print("\n📋 Engineered Dataset Overview:")
print(f"   • Date range: {df_engineered['Date'].min()} to {df_engineered['Date'].max()}")
print(f"   • Memory usage: {df_engineered.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"   • Target variable: New catches")

# Check actual column names
print(f"\n📊 Available columns: {list(df_engineered.columns)}")

# Quick peek at the data structure
display(df_engineered.head())
print("\n📊 Dataset Info:")
df_engineered.info()

📖 Loading datasets...
✅ Engineered dataset loaded: 245 rows × 17 columns
✅ Merged dataset loaded: 245 rows × 9 columns

📋 Engineered Dataset Overview:
   • Date range: 2024-07-06 to 2024-08-23
   • Memory usage: 0.1 MB
   • Target variable: New catches

📊 Available columns: ['Date', 'Location', 'Location_Code', 'Average Temperature', 'Average Humidity', 'Temp_Range', 'Temp_Avg_3d', 'Humidity_Avg_3d', 'Insects_Lag1', 'Insects_Lag3', 'Recent_Activity', 'Days_Since_Cleaning', 'Month', 'Day', 'Season', 'Number of insects', 'New catches']


,Date,Location,Location_Code,Average Temperature,Average Humidity,Temp_Range,Temp_Avg_3d,Humidity_Avg_3d,Insects_Lag1,Insects_Lag3,Recent_Activity,Days_Since_Cleaning,Month,Day,Season,Number of insects,New catches
0,2024-07-06,Cicalino 1,0,22.34,72.25,1.59,22.340000,72.250000,0.0,0.0,0,0,7,6,Mid_Summer,0.0,0.0
1,2024-07-07,Cicalino 1,0,23.52,76.73,1.22,22.930000,74.490000,0.0,0.0,0,0,7,7,Mid_Summer,0.0,0.0
2,2024-07-08,Cicalino 1,0,25.67,69.14,1.87,23.843333,72.706667,0.0,0.0,0,0,7,8,Mid_Summer,0.0,0.0
3,2024-07-09,Cicalino 1,0,25.87,53.65,1.98,25.020000,66.506667,0.0,0.0,0,0,7,9,Mid_Summer,0.0,0.0
4,2024-07-10,Cicalino 1,0,26.41,58.94,1.91,25.983333,60.576667,0.0,0.0,0,0,7,10,Mid_Summer,0.0,0.0



📊 Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 245 entries, 0 to 244
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Date                 245 non-null    object 
 1   Location             245 non-null    object 
 2   Location_Code        245 non-null    int64  
 3   Average Temperature  245 non-null    float64
 4   Average Humidity     245 non-null    float64
 5   Temp_Range           245 non-null    float64
 6   Temp_Avg_3d          245 non-null    float64
 7   Humidity_Avg_3d      245 non-null    float64
 8   Insects_Lag1         245 non-null    float64
 9   Insects_Lag3         245 non-null    float64
 10  Recent_Activity      245 non-null    int64  
 11  Days_Since_Cleaning  245 non-null    int64  
 12  Month                245 non-null    int64  
 13  Day                  245 non-null    int64  
 14  Season               245 non-null    object 
 15  Number of insects    24

## ⚖️ Class Imbalance Diagnosis

Before diving into modeling, we must understand the distribution of our target variable `NEW_CATCHES`. Class imbalance is a critical factor that can severely impact model performance if not properly addressed. We'll visualize the distribution and calculate key metrics to inform our modeling strategy.

In [16]:
# First, convert to binary classification (0 = No Catch, 1+ = Catch)
print("🔄 Converting to Binary Classification...")
df_engineered['New catches'] = (df_engineered['New catches'] > 0).astype(int)

# Calculate class distribution for binary classification
class_counts = df_engineered['New catches'].value_counts().sort_index()
class_percentages = df_engineered['New catches'].value_counts(normalize=True).sort_index() * 100

print("🎯 Target Variable Distribution Analysis (Binary):")
print("=" * 50)

for class_label in class_counts.index:
    count = class_counts[class_label]
    percentage = class_percentages[class_label]
    class_name = "No Catch" if class_label == 0 else "Catch"
    print(f"   Class {class_label} ({class_name}): {count:,} samples ({percentage:.2f}%)")

# Calculate imbalance ratio
majority_class = class_counts.max()
minority_class = class_counts.min()
imbalance_ratio = majority_class / minority_class

print(f"\n📊 Imbalance Metrics:")
print(f"   • Imbalance Ratio: {imbalance_ratio:.2f}:1")
print(f"   • Majority Class: {majority_class:,} samples")
print(f"   • Minority Class: {minority_class:,} samples")

# Determine imbalance severity
if imbalance_ratio > 10:
    severity = "🔴 SEVERE"
elif imbalance_ratio > 3:
    severity = "🟡 MODERATE"
else:
    severity = "🟢 MILD"

print(f"   • Imbalance Severity: {severity}")

# Create interactive visualization
fig = px.bar(
    x=['No Catch (0)', 'Catch (1)'],
    y=class_counts.values,
    title="🎯 Target Variable Distribution: New catches (Binary)",
    labels={'x': 'Class Label', 'y': 'Number of Samples'},
    text=class_counts.values,
    color=class_counts.values,
    color_continuous_scale='Viridis'
)

# Enhance the visualization
fig.update_traces(
    texttemplate='%{text:,}<br>(%{customdata:.1f}%)',
    customdata=class_percentages.values,
    textposition='outside'
)

fig.update_layout(
    height=500,
    showlegend=False,
    title_x=0.5,
    annotations=[
        dict(
            text=f"Imbalance Ratio: {imbalance_ratio:.2f}:1 ({severity.split()[-1]})",
            xref="paper", yref="paper",
            x=0.02, y=0.98,
            showarrow=False,
            font=dict(size=12, color="red" if "SEVERE" in severity else "orange" if "MODERATE" in severity else "green")
        )
    ]
)

fig.show()

print("\n⚠️  CRITICAL INSIGHT:")
print(f"   The {severity.split()[-1].lower()} class imbalance requires robust handling through:")
print("   1. 🎯 Class weighting in all models")
print("   2. 🔧 Optimal threshold tuning")
print("   3. 📊 F1-score as primary evaluation metric")
print(f"\n✅ Binary classification setup complete!")
print(f"   • No Catch (0): Any day with 0 insects caught")
print(f"   • Catch (1): Any day with 1+ insects caught")


🔄 Converting to Binary Classification...
🎯 Target Variable Distribution Analysis (Binary):
   Class 0 (No Catch): 224 samples (91.43%)
   Class 1 (Catch): 21 samples (8.57%)

📊 Imbalance Metrics:
   • Imbalance Ratio: 10.67:1
   • Majority Class: 224 samples
   • Minority Class: 21 samples
   • Imbalance Severity: 🔴 SEVERE



⚠️  CRITICAL INSIGHT:
   The severe class imbalance requires robust handling through:
   1. 🎯 Class weighting in all models
   2. 🔧 Optimal threshold tuning
   3. 📊 F1-score as primary evaluation metric

✅ Binary classification setup complete!
   • No Catch (0): Any day with 0 insects caught
   • Catch (1): Any day with 1+ insects caught


## 🔪 Data Splitting and Weight Calculation

We perform a **chronological split** to respect the time-series nature of our data. This prevents data leakage and ensures our model evaluation reflects real-world performance. We also calculate class weights to handle the imbalance we identified above.

In [17]:
# Prepare features and target
print("🔧 Preparing features and target variable...")

# Convert Date to datetime for proper sorting
df_engineered['Date'] = pd.to_datetime(df_engineered['Date'])

# Sort data chronologically to maintain time series integrity
df_sorted = df_engineered.sort_values('Date').reset_index(drop=True)

# Identify feature columns (exclude non-feature columns and potential leakage)
exclude_cols = ['Date', 'New catches', 'Number of insects']  # Exclude target and potential leakage
feature_cols = [col for col in df_sorted.columns if col not in exclude_cols]

print(f"🚫 Excluded columns (potential leakage/non-features): {exclude_cols}")
print(f"✅ Feature columns: {feature_cols}")

X = df_sorted[feature_cols]
y = df_sorted['New catches']
dates = df_sorted['Date']

print(f"✅ Features prepared: {len(feature_cols)} columns")
print(f"✅ Target prepared: {len(y)} samples")
print(f"📅 Data sorted chronologically from {dates.min().date()} to {dates.max().date()}")

# Perform chronological train-test split (80-20)
print("\n✂️ Performing chronological train-test split...")

split_idx = int(len(X) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]
train_dates = dates.iloc[:split_idx]
test_dates = dates.iloc[split_idx:]

train_end_date = train_dates.iloc[-1].date()
test_start_date = test_dates.iloc[0].date()

print(f"📊 Training data: {len(X_train):,} samples (up to {train_end_date})")
print(f"📊 Test data: {len(X_test):,} samples (from {test_start_date})")

# Calculate class weights from training data
print("\n⚖️ Calculating class weights...")

classes = np.unique(y_train)
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

# Create class weight dictionary
class_weights = dict(zip(classes, class_weights_array))

print("✅ Class weights calculated:")
for class_label, weight in class_weights.items():
    print(f"   Class {class_label}: {weight:.3f}")

# Verify split preserves class distribution
print("\n🔍 Verifying class distribution preservation:")
train_dist = y_train.value_counts(normalize=True).sort_index() * 100
test_dist = y_test.value_counts(normalize=True).sort_index() * 100

comparison_df = pd.DataFrame({
    'Training (%)': train_dist,
    'Test (%)': test_dist
})

display(comparison_df.round(2))

# Handle categorical variables - encode them for machine learning models
print("\n🔄 Encoding categorical variables...")

# Identify categorical columns
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
print(f"📊 Categorical columns found: {categorical_cols}")

if categorical_cols:
    # Use pandas get_dummies for one-hot encoding
    X_train_encoded = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
    X_test_encoded = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)
    
    # Ensure both train and test have the same columns
    # Get all columns from training set
    train_cols = set(X_train_encoded.columns)
    test_cols = set(X_test_encoded.columns)
    
    # Add missing columns to test set (fill with 0)
    missing_in_test = train_cols - test_cols
    for col in missing_in_test:
        X_test_encoded[col] = 0
    
    # Add missing columns to train set (fill with 0)
    missing_in_train = test_cols - train_cols
    for col in missing_in_train:
        X_train_encoded[col] = 0
    
    # Reorder columns to match
    X_test_encoded = X_test_encoded[X_train_encoded.columns]
    
    print(f"✅ Encoded features: {X_train_encoded.shape[1]} columns")
    print(f"✅ Training set: {X_train_encoded.shape}")
    print(f"✅ Test set: {X_test_encoded.shape}")
    
    # Update feature sets
    X_train = X_train_encoded
    X_test = X_test_encoded
else:
    print("✅ No categorical variables found")

# Check for data leakage by examining feature correlations with target
print("\n🔍 Data leakage check - Feature-target correlations:")
# Only check numeric columns for correlation
numeric_cols = X_train.select_dtypes(include=[np.number]).columns
correlations = X_train[numeric_cols].corrwith(y_train).abs().sort_values(ascending=False)
print("Top 5 feature correlations with target:")
for feature, corr in correlations.head().items():
    print(f"   {feature}: {corr:.3f}")

print("\n🎯 Data splitting and encoding complete! Ready for model training.")

🔧 Preparing features and target variable...
🚫 Excluded columns (potential leakage/non-features): ['Date', 'New catches', 'Number of insects']
✅ Feature columns: ['Location', 'Location_Code', 'Average Temperature', 'Average Humidity', 'Temp_Range', 'Temp_Avg_3d', 'Humidity_Avg_3d', 'Insects_Lag1', 'Insects_Lag3', 'Recent_Activity', 'Days_Since_Cleaning', 'Month', 'Day', 'Season']
✅ Features prepared: 14 columns
✅ Target prepared: 245 samples
📅 Data sorted chronologically from 2024-07-06 to 2024-08-23

✂️ Performing chronological train-test split...
📊 Training data: 196 samples (up to 2024-08-14)
📊 Test data: 49 samples (from 2024-08-14)

⚖️ Calculating class weights...
✅ Class weights calculated:
   Class 0: 0.533
   Class 1: 8.167

🔍 Verifying class distribution preservation:


,Training (%),Test (%)
New catches,,
0,93.88,81.63
1,6.12,18.37



🔄 Encoding categorical variables...
📊 Categorical columns found: ['Location', 'Season']
✅ Encoded features: 17 columns
✅ Training set: (196, 17)
✅ Test set: (49, 17)

🔍 Data leakage check - Feature-target correlations:
Top 5 feature correlations with target:
   Recent_Activity: 0.557
   Insects_Lag1: 0.347
   Insects_Lag3: 0.347
   Days_Since_Cleaning: 0.329
   Location_Code: 0.302

🎯 Data splitting and encoding complete! Ready for model training.


# 🥊 Standard Classifiers Tournament

Now begins our first tournament phase! We'll pit three powerful machine learning algorithms against each other:

1. **🌳 RandomForestClassifier**: Ensemble of decision trees with voting
2. **🚀 XGBoostClassifier**: Gradient boosting with advanced optimization
3. **⚡ LightGBMClassifier**: Microsoft's efficient gradient boosting

Each contestant will undergo:
- **Hyperparameter tuning** via GridSearchCV
- **Class-weighted training** to handle imbalance
- **Optimal threshold discovery** for maximum F1-score
- **Comprehensive evaluation** on test data

Let the battle commence! ⚔️

## 🌳 Contestant 1: RandomForestClassifier

Our first competitor leverages the wisdom of crowds through ensemble learning. Random Forest builds multiple decision trees and combines their predictions, making it robust against overfitting and excellent for handling mixed data types.

In [18]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import classification_report

print("🌳 Training RandomForestClassifier...")
print("=" * 50)

# Use all available features (already cleaned of leakage)
X_train_rf = X_train.copy()
X_test_rf = X_test.copy()

print(f"📊 Using {len(X_train_rf.columns)} features for RandomForest modeling")
print(f"✅ Training set: {X_train_rf.shape}")
print(f"✅ Test set: {X_test_rf.shape}")

# Use focused hyperparameter grid for RandomForest
rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [5, 10, 15],  # Higher values to prevent overfitting on imbalanced data
    'min_samples_leaf': [2, 4, 6],     # Higher values to prevent overfitting
    'max_features': ['sqrt', 'log2']
}

total_combinations = (len(rf_param_grid['n_estimators']) * len(rf_param_grid['max_depth']) * 
                     len(rf_param_grid['min_samples_split']) * len(rf_param_grid['min_samples_leaf']) * 
                     len(rf_param_grid['max_features']))
print(f"🔧 Hyperparameter combinations to test: {total_combinations}")

# Initialize Random Forest with class weights
rf_base = RandomForestClassifier(
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# Use TimeSeriesSplit for cross-validation to respect temporal order
tscv = TimeSeriesSplit(n_splits=3)

print("🔍 Performing hyperparameter tuning with TimeSeriesSplit...")
rf_grid = GridSearchCV(
    rf_base,
    rf_param_grid,
    cv=tscv,  # Use TimeSeriesSplit instead of regular CV
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_train_rf, y_train)

print(f"✅ Best parameters found: {rf_grid.best_params_}")
print(f"🎯 Best CV F1-score: {rf_grid.best_score_:.4f}")

# Get the best model
rf_best = rf_grid.best_estimator_

# Use a separate time-based validation set for threshold tuning
# Split training data chronologically (use 80% for training, 20% for validation)
split_point = int(len(X_train_rf) * 0.8)
X_train_cv_rf = X_train_rf.iloc[:split_point]
X_val_cv_rf = X_train_rf.iloc[split_point:]
y_train_cv_rf = y_train.iloc[:split_point]
y_val_cv_rf = y_train.iloc[split_point:]

# Train on the CV training set for threshold optimization
rf_threshold_model = RandomForestClassifier(**rf_grid.best_params_, class_weight='balanced', random_state=42, n_jobs=-1)
rf_threshold_model.fit(X_train_cv_rf, y_train_cv_rf)

print(f"\n🔧 Finding optimal prediction threshold using temporal validation...")
print(f"📊 CV Training set: {X_train_cv_rf.shape}, CV Validation set: {X_val_cv_rf.shape}")

# Get prediction probabilities for threshold tuning
y_val_proba_rf = rf_threshold_model.predict_proba(X_val_cv_rf)[:, 1]

# Find optimal threshold using F1-score
thresholds = np.arange(0.1, 0.9, 0.01)
f1_scores_rf = []

for threshold in thresholds:
    y_pred_thresh = (y_val_proba_rf >= threshold).astype(int)
    if len(np.unique(y_pred_thresh)) > 1:  # Avoid division by zero
        f1 = f1_score(y_val_cv_rf, y_pred_thresh)
    else:
        f1 = 0.0
    f1_scores_rf.append(f1)

optimal_threshold_rf = thresholds[np.argmax(f1_scores_rf)]
max_f1_rf = max(f1_scores_rf)

print(f"🎯 Optimal threshold: {optimal_threshold_rf:.3f}")
print(f"🏆 Validation F1-score at optimal threshold: {max_f1_rf:.4f}")

# Retrain on full training set with best parameters
print("\n🔄 Retraining on full training set...")
rf_final = RandomForestClassifier(**rf_grid.best_params_, class_weight='balanced', random_state=42, n_jobs=-1)
rf_final.fit(X_train_rf, y_train)

# Make predictions on test set with optimal threshold
print("\n📊 Evaluating on test set...")
y_test_proba_rf = rf_final.predict_proba(X_test_rf)[:, 1]
y_test_pred_rf = (y_test_proba_rf >= optimal_threshold_rf).astype(int)

# Calculate metrics
rf_test_f1 = f1_score(y_test, y_test_pred_rf)
rf_test_auc = roc_auc_score(y_test, y_test_proba_rf)
rf_test_accuracy = accuracy_score(y_test, y_test_pred_rf)

print(f"🎯 Test F1-score: {rf_test_f1:.4f}")
print(f"📈 Test AUC: {rf_test_auc:.4f}")
print(f"🎯 Test Accuracy: {rf_test_accuracy:.4f}")

# Detailed classification report
print(f"\n📋 Detailed Classification Report:")
print(classification_report(y_test, y_test_pred_rf, target_names=['No Catch', 'Catch']))

# Store results for later comparison
rf_results = {
    'model': rf_final,
    'threshold': optimal_threshold_rf,
    'test_f1': rf_test_f1,
    'test_auc': rf_test_auc,
    'test_accuracy': rf_test_accuracy,
    'y_pred': y_test_pred_rf,
    'y_proba': y_test_proba_rf,
    'best_params': rf_grid.best_params_
}

print("\n✅ RandomForestClassifier training complete!")

🌳 Training RandomForestClassifier...
📊 Using 17 features for RandomForest modeling
✅ Training set: (196, 17)
✅ Test set: (49, 17)
🔧 Hyperparameter combinations to test: 216
🔍 Performing hyperparameter tuning with TimeSeriesSplit...
Fitting 3 folds for each of 216 candidates, totalling 648 fits
✅ Best parameters found: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
🎯 Best CV F1-score: 0.1212

🔧 Finding optimal prediction threshold using temporal validation...
📊 CV Training set: (156, 17), CV Validation set: (40, 17)
🎯 Optimal threshold: 0.100
🏆 Validation F1-score at optimal threshold: 0.0000

🔄 Retraining on full training set...

📊 Evaluating on test set...
🎯 Test F1-score: 0.6667
📈 Test AUC: 0.9194
🎯 Test Accuracy: 0.8163

📋 Detailed Classification Report:
              precision    recall  f1-score   support

    No Catch       1.00      0.78      0.87        40
       Catch       0.50      1.00      0.67         9

    a

### 📊 RandomForest Performance Analysis

Let's dive deep into how our Random Forest model performed with interactive visualizations.

In [19]:
# Generate comprehensive classification report
print("📋 RandomForest Classification Report:")
print("=" * 60)
print(classification_report(y_test, y_test_pred_rf, target_names=['No Catch', 'Catch']))

# Create interactive confusion matrix
cm_rf = confusion_matrix(y_test, y_test_pred_rf)

fig_cm_rf = ff.create_annotated_heatmap(
    z=cm_rf,
    x=['Predicted: No Catch', 'Predicted: Catch'],
    y=['Actual: No Catch', 'Actual: Catch'],
    annotation_text=cm_rf,
    colorscale='Blues',
    showscale=True
)

fig_cm_rf.update_layout(
    title='🌳 RandomForest Confusion Matrix',
    title_x=0.5,
    width=500,
    height=400
)

fig_cm_rf.show()

# Create interactive ROC curve
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_test_proba_rf)

fig_roc_rf = go.Figure()

fig_roc_rf.add_trace(go.Scatter(
    x=fpr_rf,
    y=tpr_rf,
    mode='lines',
    name=f'RandomForest (AUC = {rf_test_auc:.3f})',
    line=dict(color='green', width=3)
))

# Add diagonal reference line
fig_roc_rf.add_trace(go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode='lines',
    name='Random Classifier',
    line=dict(color='red', width=2, dash='dash')
))

fig_roc_rf.update_layout(
    title='🌳 RandomForest ROC Curve',
    title_x=0.5,
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    width=600,
    height=500
)

fig_roc_rf.show()

# Feature importance analysis
feature_importance = pd.DataFrame({
    'feature': X_train_rf.columns,
    'importance': rf_final.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\n🎯 Top 10 Most Important Features:")
for i, (idx, row) in enumerate(feature_importance.head(10).iterrows()):
    print(f"   {i+1}. {row['feature']}: {row['importance']:.4f}")

# Create feature importance plot
fig_importance = px.bar(
    feature_importance.head(10),
    x='importance',
    y='feature',
    orientation='h',
    title='🌳 RandomForest - Top 10 Feature Importances',
    labels={'importance': 'Feature Importance', 'feature': 'Features'}
)
fig_importance.update_layout(height=500, yaxis={'categoryorder':'total ascending'})
fig_importance.show()

print(f"\n🏆 RandomForest Tournament Summary:")
print(f"   • F1-Score: {rf_test_f1:.4f}")
print(f"   • AUC: {rf_test_auc:.4f}")
print(f"   • Optimal Threshold: {optimal_threshold_rf:.3f}")
print(f"   • Test Accuracy: {rf_test_accuracy:.4f}")
print(f"   • Best Parameters: {rf_results['best_params']}")

📋 RandomForest Classification Report:
              precision    recall  f1-score   support

    No Catch       1.00      0.78      0.87        40
       Catch       0.50      1.00      0.67         9

    accuracy                           0.82        49
   macro avg       0.75      0.89      0.77        49
weighted avg       0.91      0.82      0.84        49




🎯 Top 10 Most Important Features:
   1. Recent_Activity: 0.3403
   2. Days_Since_Cleaning: 0.1148
   3. Location_Code: 0.0998
   4. Temp_Avg_3d: 0.0703
   5. Insects_Lag1: 0.0647
   6. Average Humidity: 0.0631
   7. Day: 0.0624
   8. Average Temperature: 0.0490
   9. Temp_Range: 0.0434
   10. Humidity_Avg_3d: 0.0406



🏆 RandomForest Tournament Summary:
   • F1-Score: 0.6667
   • AUC: 0.9194
   • Optimal Threshold: 0.100
   • Test Accuracy: 0.8163
   • Best Parameters: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}


## 🚀 Contestant 2: XGBoostClassifier

Enter our second competitor: XGBoost, the gradient boosting champion! Known for its exceptional performance in machine learning competitions, XGBoost uses advanced optimization techniques and handles missing values naturally.

### 📊 XGBoost Performance Analysis

Time to analyze our gradient boosting champion's performance!

In [20]:
print("🚀 Training XGBoostClassifier...")
print("=" * 50)

# Use all available features (already cleaned of leakage)
X_train_xgb = X_train.copy()
X_test_xgb = X_test.copy()

print(f"📊 Using {len(X_train_xgb.columns)} features for XGBoost modeling")
print(f"✅ Training set: {X_train_xgb.shape}")
print(f"✅ Test set: {X_test_xgb.shape}")

# Use focused hyperparameter grid for XGBoost
xgb_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

total_combinations = (len(xgb_param_grid['n_estimators']) * len(xgb_param_grid['max_depth']) * 
                     len(xgb_param_grid['learning_rate']) * len(xgb_param_grid['subsample']) * 
                     len(xgb_param_grid['colsample_bytree']))
print(f"🔧 Hyperparameter combinations to test: {total_combinations}")

# Calculate scale_pos_weight for XGBoost (equivalent to class_weight='balanced')
scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

# Initialize XGBoost with class weights
xgb_base = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)

print("🔍 Performing hyperparameter tuning with TimeSeriesSplit...")
xgb_grid = GridSearchCV(
    xgb_base,
    xgb_param_grid,
    cv=tscv,  # Use TimeSeriesSplit
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

xgb_grid.fit(X_train_xgb, y_train)

print(f"✅ Best parameters found: {xgb_grid.best_params_}")
print(f"🎯 Best CV F1-score: {xgb_grid.best_score_:.4f}")

# Get the best model
xgb_best = xgb_grid.best_estimator_

# Train on the CV training set for threshold optimization
xgb_threshold_model = XGBClassifier(**xgb_grid.best_params_, scale_pos_weight=scale_pos_weight, random_state=42, n_jobs=-1, eval_metric='logloss')
xgb_threshold_model.fit(X_train_cv_rf, y_train_cv_rf)

print(f"\n🔧 Finding optimal prediction threshold using temporal validation...")

# Get prediction probabilities for threshold tuning
y_val_proba_xgb = xgb_threshold_model.predict_proba(X_val_cv_rf)[:, 1]

# Find optimal threshold using F1-score
f1_scores_xgb = []

for threshold in thresholds:
    y_pred_thresh = (y_val_proba_xgb >= threshold).astype(int)
    if len(np.unique(y_pred_thresh)) > 1:  # Avoid division by zero
        f1 = f1_score(y_val_cv_rf, y_pred_thresh)
    else:
        f1 = 0.0
    f1_scores_xgb.append(f1)

optimal_threshold_xgb = thresholds[np.argmax(f1_scores_xgb)]
max_f1_xgb = max(f1_scores_xgb)

print(f"🎯 Optimal threshold: {optimal_threshold_xgb:.3f}")
print(f"🏆 Validation F1-score at optimal threshold: {max_f1_xgb:.4f}")

# Retrain on full training set with best parameters
print("\n🔄 Retraining on full training set...")
xgb_final = XGBClassifier(**xgb_grid.best_params_, scale_pos_weight=scale_pos_weight, random_state=42, n_jobs=-1, eval_metric='logloss')
xgb_final.fit(X_train_xgb, y_train)

# Make predictions on test set with optimal threshold
print("\n📊 Evaluating on test set...")
y_test_proba_xgb = xgb_final.predict_proba(X_test_xgb)[:, 1]
y_test_pred_xgb = (y_test_proba_xgb >= optimal_threshold_xgb).astype(int)

# Calculate metrics
xgb_test_f1 = f1_score(y_test, y_test_pred_xgb)
xgb_test_auc = roc_auc_score(y_test, y_test_proba_xgb)
xgb_test_accuracy = accuracy_score(y_test, y_test_pred_xgb)

print(f"🎯 Test F1-score: {xgb_test_f1:.4f}")
print(f"📈 Test AUC: {xgb_test_auc:.4f}")
print(f"🎯 Test Accuracy: {xgb_test_accuracy:.4f}")

# Store results for later comparison
xgb_results = {
    'model': xgb_final,
    'threshold': optimal_threshold_xgb,
    'test_f1': xgb_test_f1,
    'test_auc': xgb_test_auc,
    'test_accuracy': xgb_test_accuracy,
    'y_pred': y_test_pred_xgb,
    'y_proba': y_test_proba_xgb,
    'best_params': xgb_grid.best_params_
}

print("\n✅ XGBoostClassifier training complete!")

# Generate comprehensive classification report
print("📋 XGBoost Classification Report:")
print("=" * 60)
print(classification_report(y_test, y_test_pred_xgb, target_names=['No Catch', 'Catch']))

# Create interactive confusion matrix
cm_xgb = confusion_matrix(y_test, y_test_pred_xgb)

fig_cm_xgb = ff.create_annotated_heatmap(
    z=cm_xgb,
    x=['Predicted: No Catch', 'Predicted: Catch'],
    y=['Actual: No Catch', 'Actual: Catch'],
    annotation_text=cm_xgb,
    colorscale='Oranges',
    showscale=True
)

fig_cm_xgb.update_layout(
    title='🚀 XGBoost Confusion Matrix',
    title_x=0.5,
    width=500,
    height=400
)

fig_cm_xgb.show()

# Create interactive ROC curve
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_test_proba_xgb)

fig_roc_xgb = go.Figure()

fig_roc_xgb.add_trace(go.Scatter(
    x=fpr_xgb,
    y=tpr_xgb,
    mode='lines',
    name=f'XGBoost (AUC = {xgb_test_auc:.3f})',
    line=dict(color='orange', width=3)
))

# Add diagonal reference line
fig_roc_xgb.add_trace(go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode='lines',
    name='Random Classifier',
    line=dict(color='red', width=2, dash='dash')
))

fig_roc_xgb.update_layout(
    title='🚀 XGBoost ROC Curve',
    title_x=0.5,
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    width=600,
    height=500
)

fig_roc_xgb.show()

# Feature importance analysis for XGBoost
feature_importance_xgb = pd.DataFrame({
    'feature': X_train_xgb.columns,
    'importance': xgb_final.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\n🎯 XGBoost - Top 10 Most Important Features:")
for i, (idx, row) in enumerate(feature_importance_xgb.head(10).iterrows()):
    print(f"   {i+1}. {row['feature']}: {row['importance']:.4f}")

# Create feature importance plot
fig_importance_xgb = px.bar(
    feature_importance_xgb.head(10),
    x='importance',
    y='feature',
    orientation='h',
    title='🚀 XGBoost - Top 10 Feature Importances',
    labels={'importance': 'Feature Importance', 'feature': 'Features'},
    color_discrete_sequence=['orange']
)
fig_importance_xgb.update_layout(height=500, yaxis={'categoryorder':'total ascending'})
fig_importance_xgb.show()

print(f"\n🏆 XGBoost Tournament Summary:")
print(f"   • F1-Score: {xgb_test_f1:.4f}")
print(f"   • AUC: {xgb_test_auc:.4f}")
print(f"   • Optimal Threshold: {optimal_threshold_xgb:.3f}")
print(f"   • Test Accuracy: {xgb_test_accuracy:.4f}")
print(f"   • Best Parameters: {xgb_results['best_params']}")

🚀 Training XGBoostClassifier...
📊 Using 17 features for XGBoost modeling
✅ Training set: (196, 17)
✅ Test set: (49, 17)
🔧 Hyperparameter combinations to test: 108
🔍 Performing hyperparameter tuning with TimeSeriesSplit...
Fitting 3 folds for each of 108 candidates, totalling 324 fits
✅ Best parameters found: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100, 'subsample': 1.0}
🎯 Best CV F1-score: 0.4563

🔧 Finding optimal prediction threshold using temporal validation...
🎯 Optimal threshold: 0.100
🏆 Validation F1-score at optimal threshold: 0.0000

🔄 Retraining on full training set...

📊 Evaluating on test set...
🎯 Test F1-score: 0.3103
📈 Test AUC: 0.9319
🎯 Test Accuracy: 0.1837

✅ XGBoostClassifier training complete!
📋 XGBoost Classification Report:
              precision    recall  f1-score   support

    No Catch       0.00      0.00      0.00        40
       Catch       0.18      1.00      0.31         9

    accuracy                           0.


🎯 XGBoost - Top 10 Most Important Features:
   1. Recent_Activity: 0.4746
   2. Location_Code: 0.2509
   3. Days_Since_Cleaning: 0.1385
   4. Day: 0.0445
   5. Average Humidity: 0.0334
   6. Temp_Avg_3d: 0.0274
   7. Humidity_Avg_3d: 0.0110
   8. Temp_Range: 0.0103
   9. Average Temperature: 0.0051
   10. Month: 0.0042



🏆 XGBoost Tournament Summary:
   • F1-Score: 0.3103
   • AUC: 0.9319
   • Optimal Threshold: 0.100
   • Test Accuracy: 0.1837
   • Best Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100, 'subsample': 1.0}


In [21]:
print("🚀 Training XGBoostClassifier...")
print("=" * 50)

# Use all available features for XGBoost
X_train_xgb = X_train.copy()
X_test_xgb = X_test.copy()

print(f"📊 Using {len(X_train_xgb.columns)} features for XGBoost modeling")
print(f"✅ Training set: {X_train_xgb.shape}")
print(f"✅ Test set: {X_test_xgb.shape}")

# Calculate scale_pos_weight for XGBoost (equivalent to class_weight='balanced')
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count

print(f"⚖️ Scale positive weight: {scale_pos_weight:.3f}")

# Define XGBoost-specific hyperparameter grid
xgb_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 6],              # Different from RandomForest
    'learning_rate': [0.01, 0.1, 0.2],   # XGBoost-specific
    'subsample': [0.8, 0.9, 1.0],        # XGBoost-specific
    'colsample_bytree': [0.8, 0.9, 1.0], # XGBoost-specific
    'reg_alpha': [0, 0.1],               # L1 regularization
    'reg_lambda': [1, 1.1]               # L2 regularization
}

total_combinations_xgb = (len(xgb_param_grid['n_estimators']) * len(xgb_param_grid['max_depth']) * 
                         len(xgb_param_grid['learning_rate']) * len(xgb_param_grid['subsample']) * 
                         len(xgb_param_grid['colsample_bytree']) * len(xgb_param_grid['reg_alpha']) * 
                         len(xgb_param_grid['reg_lambda']))
print(f"🔧 Hyperparameter combinations to test: {total_combinations_xgb}")

# Initialize XGBoost with class balancing
xgb_base = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)

# Use TimeSeriesSplit for cross-validation
print("🔍 Performing hyperparameter tuning with TimeSeriesSplit...")
xgb_grid = GridSearchCV(
    xgb_base,
    xgb_param_grid,
    cv=tscv,  # Use TimeSeriesSplit
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

xgb_grid.fit(X_train_xgb, y_train)

print(f"✅ Best parameters found: {xgb_grid.best_params_}")
print(f"🎯 Best CV F1-score: {xgb_grid.best_score_:.4f}")

# Get the best model
xgb_best = xgb_grid.best_estimator_

# Use separate validation set for threshold tuning
split_point_xgb = int(len(X_train_xgb) * 0.8)
X_train_cv_xgb = X_train_xgb.iloc[:split_point_xgb]
X_val_cv_xgb = X_train_xgb.iloc[split_point_xgb:]
y_train_cv_xgb = y_train.iloc[:split_point_xgb]
y_val_cv_xgb = y_train.iloc[split_point_xgb:]

# Train threshold optimization model
xgb_threshold_model = XGBClassifier(**xgb_grid.best_params_, scale_pos_weight=scale_pos_weight, 
                                   random_state=42, eval_metric='logloss', verbosity=0)
xgb_threshold_model.fit(X_train_cv_xgb, y_train_cv_xgb)

print(f"\n🔧 Finding optimal prediction threshold for XGBoost...")
print(f"📊 CV Training set: {X_train_cv_xgb.shape}, CV Validation set: {X_val_cv_xgb.shape}")

# Get prediction probabilities
y_val_proba_xgb = xgb_threshold_model.predict_proba(X_val_cv_xgb)[:, 1]

# Find optimal threshold
thresholds = np.arange(0.1, 0.9, 0.01)
f1_scores_xgb = []

for threshold in thresholds:
    y_pred_thresh = (y_val_proba_xgb >= threshold).astype(int)
    if len(np.unique(y_pred_thresh)) > 1:
        f1 = f1_score(y_val_cv_xgb, y_pred_thresh)
    else:
        f1 = 0.0
    f1_scores_xgb.append(f1)

optimal_threshold_xgb = thresholds[np.argmax(f1_scores_xgb)]
max_f1_xgb = max(f1_scores_xgb)

print(f"🎯 Optimal threshold: {optimal_threshold_xgb:.3f}")
print(f"🏆 Validation F1-score at optimal threshold: {max_f1_xgb:.4f}")

# Retrain on full training set
print("\n🔄 Retraining XGBoost on full training set...")
xgb_final = XGBClassifier(**xgb_grid.best_params_, scale_pos_weight=scale_pos_weight, 
                         random_state=42, eval_metric='logloss', verbosity=0)
xgb_final.fit(X_train_xgb, y_train)

# Make predictions on test set
print("\n📊 Evaluating XGBoost on test set...")
y_test_proba_xgb = xgb_final.predict_proba(X_test_xgb)[:, 1]
y_test_pred_xgb = (y_test_proba_xgb >= optimal_threshold_xgb).astype(int)

# Calculate metrics
xgb_test_f1 = f1_score(y_test, y_test_pred_xgb)
xgb_test_auc = roc_auc_score(y_test, y_test_proba_xgb)
xgb_test_accuracy = accuracy_score(y_test, y_test_pred_xgb)

print(f"🎯 Test F1-score: {xgb_test_f1:.4f}")
print(f"📈 Test AUC: {xgb_test_auc:.4f}")
print(f"🎯 Test Accuracy: {xgb_test_accuracy:.4f}")

# Detailed classification report
print(f"\n📋 Detailed Classification Report:")
print(classification_report(y_test, y_test_pred_xgb, target_names=['No Catch', 'Catch']))

# Store results for comparison
xgb_results = {
    'model': xgb_final,
    'threshold': optimal_threshold_xgb,
    'test_f1': xgb_test_f1,
    'test_auc': xgb_test_auc,
    'test_accuracy': xgb_test_accuracy,
    'y_pred': y_test_pred_xgb,
    'y_proba': y_test_proba_xgb,
    'best_params': xgb_grid.best_params_
}

print("\n✅ XGBoostClassifier training complete!")

🚀 Training XGBoostClassifier...
📊 Using 17 features for XGBoost modeling
✅ Training set: (196, 17)
✅ Test set: (49, 17)
⚖️ Scale positive weight: 15.333
🔧 Hyperparameter combinations to test: 972
🔍 Performing hyperparameter tuning with TimeSeriesSplit...
Fitting 3 folds for each of 972 candidates, totalling 2916 fits
✅ Best parameters found: {'colsample_bytree': 0.9, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200, 'reg_alpha': 0, 'reg_lambda': 1.1, 'subsample': 1.0}
🎯 Best CV F1-score: 0.4722

🔧 Finding optimal prediction threshold for XGBoost...
📊 CV Training set: (156, 17), CV Validation set: (40, 17)
🎯 Optimal threshold: 0.100
🏆 Validation F1-score at optimal threshold: 0.0000

🔄 Retraining XGBoost on full training set...

📊 Evaluating XGBoost on test set...
🎯 Test F1-score: 0.4737
📈 Test AUC: 0.9319
🎯 Test Accuracy: 0.5918

📋 Detailed Classification Report:
              precision    recall  f1-score   support

    No Catch       1.00      0.50      0.67        40
     

### 📊 XGBoost Performance Analysis

Time to analyze our gradient boosting champion's performance!

In [22]:
# Generate comprehensive classification report
print("📋 XGBoost Classification Report:")
print("=" * 60)
print(classification_report(y_test, y_test_pred_xgb, target_names=['No Pest', 'Pest Detected']))

# Create interactive confusion matrix
cm_xgb = confusion_matrix(y_test, y_test_pred_xgb)

fig_cm_xgb = ff.create_annotated_heatmap(
    z=cm_xgb,
    x=['Predicted: No Pest', 'Predicted: Pest'],
    y=['Actual: No Pest', 'Actual: Pest'],
    annotation_text=cm_xgb,
    colorscale='Oranges',
    showscale=True
)

fig_cm_xgb.update_layout(
    title='🚀 XGBoost Confusion Matrix',
    title_x=0.5,
    width=500,
    height=400
)

fig_cm_xgb.show()

# Create interactive ROC curve
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_test_proba_xgb)

fig_roc_xgb = go.Figure()

fig_roc_xgb.add_trace(go.Scatter(
    x=fpr_xgb,
    y=tpr_xgb,
    mode='lines',
    name=f'XGBoost (AUC = {xgb_test_auc:.3f})',
    line=dict(color='orange', width=3)
))

# Add diagonal reference line
fig_roc_xgb.add_trace(go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode='lines',
    name='Random Classifier',
    line=dict(color='red', width=2, dash='dash')
))

fig_roc_xgb.update_layout(
    title='🚀 XGBoost ROC Curve',
    title_x=0.5,
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    width=600,
    height=500
)

fig_roc_xgb.show()

print(f"\n🏆 XGBoost Tournament Summary:")
print(f"   • F1-Score: {xgb_test_f1:.4f}")
print(f"   • AUC: {xgb_test_auc:.4f}")
print(f"   • Optimal Threshold: {optimal_threshold_xgb:.3f}")
print(f"   • Test Accuracy: {xgb_test_accuracy:.4f}")

📋 XGBoost Classification Report:
               precision    recall  f1-score   support

      No Pest       1.00      0.50      0.67        40
Pest Detected       0.31      1.00      0.47         9

     accuracy                           0.59        49
    macro avg       0.66      0.75      0.57        49
 weighted avg       0.87      0.59      0.63        49




🏆 XGBoost Tournament Summary:
   • F1-Score: 0.4737
   • AUC: 0.9319
   • Optimal Threshold: 0.100
   • Test Accuracy: 0.5918


## ⚡ Contestant 3: LightGBMClassifier

Our final traditional ML competitor enters the arena! LightGBM, Microsoft's lightning-fast gradient boosting framework, promises efficiency without sacrificing performance. Known for its speed and memory efficiency.

In [23]:
print("⚡ Training LightGBMClassifier...")
print("=" * 50)

# Create numeric datasets for LightGBM (ensure same features as other models)
# Use the same approach as RandomForest and XGBoost to ensure consistency
X_train_numeric = X_train_encoded.select_dtypes(include=[np.number])
X_test_numeric = X_test_encoded.select_dtypes(include=[np.number])

# Ensure both datasets have the same columns
train_cols = set(X_train_numeric.columns)
test_cols = set(X_test_numeric.columns)

# Add missing columns to test set with zeros
missing_in_test = train_cols - test_cols
for col in missing_in_test:
    X_test_numeric[col] = 0

# Add missing columns to train set with zeros (if any)
missing_in_train = test_cols - train_cols
for col in missing_in_train:
    X_train_numeric[col] = 0

# Ensure columns are in the same order
X_test_numeric = X_test_numeric[X_train_numeric.columns]

X_train_cv = X_train_numeric.iloc[:split_point]
X_val_cv = X_train_numeric.iloc[split_point:]
y_train_cv = y_train.iloc[:split_point]
y_val_cv = y_train.iloc[split_point:]

print(f"📊 Training set shape: {X_train_numeric.shape}")
print(f"📊 Test set shape: {X_test_numeric.shape}")
print(f"📊 CV Training set: {X_train_cv.shape}, CV Validation set: {X_val_cv.shape}")

# Use smaller hyperparameter grid due to severe imbalance and computational efficiency
lgb_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10],
    'learning_rate': [0.01, 0.1],
    'num_leaves': [20, 31],
    'subsample': [0.8, 1.0]
}

print(f"🔧 Hyperparameter combinations to test: {len(lgb_param_grid['n_estimators']) * len(lgb_param_grid['max_depth']) * len(lgb_param_grid['learning_rate']) * len(lgb_param_grid['num_leaves']) * len(lgb_param_grid['subsample'])}")

# Initialize LightGBM with class balancing
lgb_base = LGBMClassifier(
    class_weight='balanced',
    random_state=42,
    verbosity=-1,
    force_col_wise=True
)

# Use TimeSeriesSplit for cross-validation to respect temporal order
print("🔍 Performing hyperparameter tuning with TimeSeriesSplit...")
lgb_grid = GridSearchCV(
    lgb_base,
    lgb_param_grid,
    cv=tscv,  # Use TimeSeriesSplit instead of regular CV
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

lgb_grid.fit(X_train_numeric, y_train)

print(f"✅ Best parameters found: {lgb_grid.best_params_}")
print(f"🎯 Best CV F1-score: {lgb_grid.best_score_:.4f}")

# Get the best model
lgb_best = lgb_grid.best_estimator_

# Use the same temporal validation approach as RandomForest and XGBoost
print(f"\n🔧 Finding optimal prediction threshold using temporal validation...")
print(f"📊 CV Training set: {X_train_cv.shape}, CV Validation set: {X_val_cv.shape}")

# Train on the CV training set
lgb_best.fit(X_train_cv, y_train_cv)

# Get prediction probabilities for threshold tuning on validation set
y_val_proba_lgb = lgb_best.predict_proba(X_val_cv)[:, 1]

# Find optimal threshold using validation set
thresholds = np.arange(0.1, 0.9, 0.01)
f1_scores_lgb = []

for threshold in thresholds:
    y_pred_thresh = (y_val_proba_lgb >= threshold).astype(int)
    if len(np.unique(y_pred_thresh)) > 1:  # Avoid division by zero
        f1 = f1_score(y_val_cv, y_pred_thresh)
    else:
        f1 = 0.0
    f1_scores_lgb.append(f1)

optimal_threshold_lgb = thresholds[np.argmax(f1_scores_lgb)]
max_f1_lgb = max(f1_scores_lgb)

print(f"🎯 Optimal threshold: {optimal_threshold_lgb:.3f}")
print(f"🏆 Validation F1-score at optimal threshold: {max_f1_lgb:.4f}")

# Retrain on full training set with best parameters
print("\n🔄 Retraining on full training set...")
lgb_final = LGBMClassifier(**lgb_grid.best_params_, class_weight='balanced', random_state=42, verbosity=-1, force_col_wise=True)
lgb_final.fit(X_train_numeric, y_train)

# Make predictions on test set with optimal threshold
print("\n📊 Evaluating on test set...")
y_test_proba_lgb = lgb_final.predict_proba(X_test_numeric)[:, 1]
y_test_pred_lgb = (y_test_proba_lgb >= optimal_threshold_lgb).astype(int)

# Calculate metrics
lgb_test_f1 = f1_score(y_test, y_test_pred_lgb)
lgb_test_auc = roc_auc_score(y_test, y_test_proba_lgb)
lgb_test_accuracy = accuracy_score(y_test, y_test_pred_lgb)

print(f"🎯 Test F1-score: {lgb_test_f1:.4f}")
print(f"📈 Test AUC: {lgb_test_auc:.4f}")
print(f"🎯 Test Accuracy: {lgb_test_accuracy:.4f}")

# Detailed classification report
print(f"\n📋 Detailed Classification Report:")
print(classification_report(y_test, y_test_pred_lgb, target_names=['No Catch', 'Catch']))

# Store results for later comparison
lgb_results = {
    'model': lgb_final,
    'threshold': optimal_threshold_lgb,
    'test_f1': lgb_test_f1,
    'test_auc': lgb_test_auc,
    'test_accuracy': lgb_test_accuracy,
    'y_pred': y_test_pred_lgb,
    'y_proba': y_test_proba_lgb
}

print("\n✅ LightGBMClassifier training complete!")

⚡ Training LightGBMClassifier...
📊 Training set shape: (196, 13)
📊 Test set shape: (49, 13)
📊 CV Training set: (156, 13), CV Validation set: (40, 13)
🔧 Hyperparameter combinations to test: 32
🔍 Performing hyperparameter tuning with TimeSeriesSplit...
Fitting 3 folds for each of 32 candidates, totalling 96 fits
✅ Best parameters found: {'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 200, 'num_leaves': 20, 'subsample': 0.8}
🎯 Best CV F1-score: 0.2165

🔧 Finding optimal prediction threshold using temporal validation...
📊 CV Training set: (156, 13), CV Validation set: (40, 13)
🎯 Optimal threshold: 0.100
🏆 Validation F1-score at optimal threshold: 0.0000

🔄 Retraining on full training set...

📊 Evaluating on test set...
🎯 Test F1-score: 0.7500
📈 Test AUC: 0.9375
🎯 Test Accuracy: 0.8776

📋 Detailed Classification Report:
              precision    recall  f1-score   support

    No Catch       1.00      0.85      0.92        40
       Catch       0.60      1.00      0.75         9

 

### 📊 LightGBM Performance Analysis

Let's examine our speed demon's performance metrics!

In [24]:
print("⚡ Training LightGBMClassifier...")
print("=" * 50)

# Use all available features for LightGBM
X_train_lgb = X_train.copy()
X_test_lgb = X_test.copy()

print(f"📊 Using {len(X_train_lgb.columns)} features for LightGBM modeling")
print(f"✅ Training set: {X_train_lgb.shape}")
print(f"✅ Test set: {X_test_lgb.shape}")

# Define LightGBM-specific hyperparameter grid
lgb_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15],             # Different range from others
    'learning_rate': [0.01, 0.05, 0.1],   # Different from XGBoost
    'num_leaves': [15, 31, 50],           # LightGBM-specific parameter
    'subsample': [0.7, 0.8, 1.0],        # Different range
    'colsample_bytree': [0.7, 0.8, 1.0],  # Different range
    'min_child_samples': [10, 20, 30]     # LightGBM-specific
}

total_combinations_lgb = (len(lgb_param_grid['n_estimators']) * len(lgb_param_grid['max_depth']) * 
                         len(lgb_param_grid['learning_rate']) * len(lgb_param_grid['num_leaves']) * 
                         len(lgb_param_grid['subsample']) * len(lgb_param_grid['colsample_bytree']) * 
                         len(lgb_param_grid['min_child_samples']))
print(f"🔧 Hyperparameter combinations to test: {total_combinations_lgb}")

# Initialize LightGBM with class balancing
lgb_base = LGBMClassifier(
    class_weight='balanced',
    random_state=42,
    verbosity=-1,
    force_col_wise=True
)

# Use TimeSeriesSplit for cross-validation
print("🔍 Performing hyperparameter tuning with TimeSeriesSplit...")
lgb_grid = GridSearchCV(
    lgb_base,
    lgb_param_grid,
    cv=tscv,  # Use TimeSeriesSplit
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

lgb_grid.fit(X_train_lgb, y_train)

print(f"✅ Best parameters found: {lgb_grid.best_params_}")
print(f"🎯 Best CV F1-score: {lgb_grid.best_score_:.4f}")

# Get the best model
lgb_best = lgb_grid.best_estimator_

# Use separate validation set for threshold tuning
split_point_lgb = int(len(X_train_lgb) * 0.8)
X_train_cv_lgb = X_train_lgb.iloc[:split_point_lgb]
X_val_cv_lgb = X_train_lgb.iloc[split_point_lgb:]
y_train_cv_lgb = y_train.iloc[:split_point_lgb]
y_val_cv_lgb = y_train.iloc[split_point_lgb:]

# Train threshold optimization model
lgb_threshold_model = LGBMClassifier(**lgb_grid.best_params_, class_weight='balanced', 
                                    random_state=42, verbosity=-1, force_col_wise=True)
lgb_threshold_model.fit(X_train_cv_lgb, y_train_cv_lgb)

print(f"\n🔧 Finding optimal prediction threshold for LightGBM...")
print(f"📊 CV Training set: {X_train_cv_lgb.shape}, CV Validation set: {X_val_cv_lgb.shape}")

# Get prediction probabilities
y_val_proba_lgb = lgb_threshold_model.predict_proba(X_val_cv_lgb)[:, 1]

# Find optimal threshold
thresholds = np.arange(0.1, 0.9, 0.01)
f1_scores_lgb = []

for threshold in thresholds:
    y_pred_thresh = (y_val_proba_lgb >= threshold).astype(int)
    if len(np.unique(y_pred_thresh)) > 1:
        f1 = f1_score(y_val_cv_lgb, y_pred_thresh)
    else:
        f1 = 0.0
    f1_scores_lgb.append(f1)

optimal_threshold_lgb = thresholds[np.argmax(f1_scores_lgb)]
max_f1_lgb = max(f1_scores_lgb)

print(f"🎯 Optimal threshold: {optimal_threshold_lgb:.3f}")
print(f"🏆 Validation F1-score at optimal threshold: {max_f1_lgb:.4f}")

# Retrain on full training set
print("\n🔄 Retraining LightGBM on full training set...")
lgb_final = LGBMClassifier(**lgb_grid.best_params_, class_weight='balanced', 
                          random_state=42, verbosity=-1, force_col_wise=True)
lgb_final.fit(X_train_lgb, y_train)

# Make predictions on test set
print("\n📊 Evaluating LightGBM on test set...")
y_test_proba_lgb = lgb_final.predict_proba(X_test_lgb)[:, 1]
y_test_pred_lgb = (y_test_proba_lgb >= optimal_threshold_lgb).astype(int)

# Calculate metrics
lgb_test_f1 = f1_score(y_test, y_test_pred_lgb)
lgb_test_auc = roc_auc_score(y_test, y_test_proba_lgb)
lgb_test_accuracy = accuracy_score(y_test, y_test_pred_lgb)

print(f"🎯 Test F1-score: {lgb_test_f1:.4f}")
print(f"📈 Test AUC: {lgb_test_auc:.4f}")
print(f"🎯 Test Accuracy: {lgb_test_accuracy:.4f}")

# Detailed classification report
print(f"\n📋 Detailed Classification Report:")
print(classification_report(y_test, y_test_pred_lgb, target_names=['No Catch', 'Catch']))

# Store results for comparison
lgb_results = {
    'model': lgb_final,
    'threshold': optimal_threshold_lgb,
    'test_f1': lgb_test_f1,
    'test_auc': lgb_test_auc,
    'test_accuracy': lgb_test_accuracy,
    'y_pred': y_test_pred_lgb,
    'y_proba': y_test_proba_lgb,
    'best_params': lgb_grid.best_params_
}

print("\n✅ LightGBMClassifier training complete!")

# Generate comprehensive classification report
print("\n📋 LightGBM Classification Report:")
print("=" * 60)
print(classification_report(y_test, y_test_pred_lgb, target_names=['No Catch', 'Catch']))

# Create interactive confusion matrix
cm_lgb = confusion_matrix(y_test, y_test_pred_lgb)

fig_cm_lgb = ff.create_annotated_heatmap(
    z=cm_lgb,
    x=['Predicted: No Catch', 'Predicted: Catch'],
    y=['Actual: No Catch', 'Actual: Catch'],
    annotation_text=cm_lgb,
    colorscale='Purples',
    showscale=True
)

fig_cm_lgb.update_layout(
    title='⚡ LightGBM Confusion Matrix',
    title_x=0.5,
    width=500,
    height=400
)

fig_cm_lgb.show()

# Create interactive ROC curve
fpr_lgb, tpr_lgb, _ = roc_curve(y_test, y_test_proba_lgb)

fig_roc_lgb = go.Figure()

fig_roc_lgb.add_trace(go.Scatter(
    x=fpr_lgb,
    y=tpr_lgb,
    mode='lines',
    name=f'LightGBM (AUC = {lgb_test_auc:.3f})',
    line=dict(color='purple', width=3)
))

# Add diagonal reference line
fig_roc_lgb.add_trace(go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode='lines',
    name='Random Classifier',
    line=dict(color='red', width=2, dash='dash')
))

fig_roc_lgb.update_layout(
    title='⚡ LightGBM ROC Curve',
    title_x=0.5,
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    width=600,
    height=500
)

fig_roc_lgb.show()

# Feature importance analysis for LightGBM
feature_importance_lgb = pd.DataFrame({
    'feature': X_train_lgb.columns,
    'importance': lgb_final.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\n🎯 LightGBM - Top 10 Most Important Features:")
for i, (idx, row) in enumerate(feature_importance_lgb.head(10).iterrows()):
    print(f"   {i+1}. {row['feature']}: {row['importance']:.4f}")

# Create feature importance plot
fig_importance_lgb = px.bar(
    feature_importance_lgb.head(10),
    x='importance',
    y='feature',
    orientation='h',
    title='⚡ LightGBM - Top 10 Feature Importances',
    labels={'importance': 'Feature Importance', 'feature': 'Features'},
    color_discrete_sequence=['purple']
)
fig_importance_lgb.update_layout(height=500, yaxis={'categoryorder':'total ascending'})
fig_importance_lgb.show()

print(f"\n🏆 LightGBM Tournament Summary:")
print(f"   • F1-Score: {lgb_test_f1:.4f}")
print(f"   • AUC: {lgb_test_auc:.4f}")
print(f"   • Optimal Threshold: {optimal_threshold_lgb:.3f}")
print(f"   • Test Accuracy: {lgb_test_accuracy:.4f}")
print(f"   • Best Parameters: {lgb_results['best_params']}")

⚡ Training LightGBMClassifier...
📊 Using 17 features for LightGBM modeling
✅ Training set: (196, 17)
✅ Test set: (49, 17)
🔧 Hyperparameter combinations to test: 2187
🔍 Performing hyperparameter tuning with TimeSeriesSplit...
Fitting 3 folds for each of 2187 candidates, totalling 6561 fits
✅ Best parameters found: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 5, 'min_child_samples': 20, 'n_estimators': 300, 'num_leaves': 15, 'subsample': 0.7}
🎯 Best CV F1-score: 0.2343

🔧 Finding optimal prediction threshold for LightGBM...
📊 CV Training set: (156, 17), CV Validation set: (40, 17)
🎯 Optimal threshold: 0.100
🏆 Validation F1-score at optimal threshold: 0.0000

🔄 Retraining LightGBM on full training set...

📊 Evaluating LightGBM on test set...
🎯 Test F1-score: 0.5625
📈 Test AUC: 0.9250
🎯 Test Accuracy: 0.7143

📋 Detailed Classification Report:
              precision    recall  f1-score   support

    No Catch       1.00      0.65      0.79        40
       Catch       0.39


🎯 LightGBM - Top 10 Most Important Features:
   1. Day: 466.0000
   2. Recent_Activity: 237.0000
   3. Location_Code: 208.0000
   4. Temp_Range: 192.0000
   5. Average Temperature: 172.0000
   6. Temp_Avg_3d: 165.0000
   7. Average Humidity: 159.0000
   8. Days_Since_Cleaning: 133.0000
   9. Month: 77.0000
   10. Season_Mid_Summer: 65.0000



🏆 LightGBM Tournament Summary:
   • F1-Score: 0.5625
   • AUC: 0.9250
   • Optimal Threshold: 0.100
   • Test Accuracy: 0.7143
   • Best Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 5, 'min_child_samples': 20, 'n_estimators': 300, 'num_leaves': 15, 'subsample': 0.7}


# 🏆 Standard Classifiers Champion Selection

The moment of truth has arrived! Let's compare all three contestants and crown our **Standard Classifier Champion**. We'll use F1-score as our primary metric (perfect for imbalanced datasets) and AUC as our secondary metric.

In [25]:
print("🏆 STANDARD CLASSIFIERS TOURNAMENT RESULTS")
print("=" * 70)

# Create comprehensive comparison table
comparison_data = {
    'Model': ['🌳 RandomForest', '🚀 XGBoost', '⚡ LightGBM'],
    'F1-Score': [rf_results['test_f1'], xgb_results['test_f1'], lgb_results['test_f1']],
    'AUC': [rf_results['test_auc'], xgb_results['test_auc'], lgb_results['test_auc']],
    'Accuracy': [rf_results['test_accuracy'], xgb_results['test_accuracy'], lgb_results['test_accuracy']],
    'Optimal Threshold': [rf_results['threshold'], xgb_results['threshold'], lgb_results['threshold']]
}

comparison_df = pd.DataFrame(comparison_data)

# Sort by F1-Score (primary) and AUC (secondary)
comparison_df = comparison_df.sort_values(['F1-Score', 'AUC'], ascending=False)

print("📊 Final Standings:")
display(comparison_df.round(4))

# Determine champion
champion_idx = comparison_df.index[0]
champion_name = comparison_df.iloc[0]['Model']
champion_f1 = comparison_df.iloc[0]['F1-Score']
champion_auc = comparison_df.iloc[0]['AUC']

print(f"\n🥇 STANDARD CLASSIFIER CHAMPION: {champion_name}")
print(f"   • Winning F1-Score: {champion_f1:.4f}")
print(f"   • Winning AUC: {champion_auc:.4f}")

# Store champion for later use
if 'RandomForest' in champion_name:
    standard_champion = rf_results
    standard_champion_name = 'RandomForest'
elif 'XGBoost' in champion_name:
    standard_champion = xgb_results
    standard_champion_name = 'XGBoost'
else:
    standard_champion = lgb_results
    standard_champion_name = 'LightGBM'

# Create interactive comparison visualization
fig_comparison = make_subplots(
    rows=1, cols=2,
    subplot_titles=['F1-Score Comparison', 'AUC Comparison'],
    specs=[[{"type": "bar"}, {"type": "bar"}]]
)

models = ['RandomForest', 'XGBoost', 'LightGBM']
colors = ['green', 'orange', 'purple']
f1_scores = [rf_results['test_f1'], xgb_results['test_f1'], lgb_results['test_f1']]
auc_scores = [rf_results['test_auc'], xgb_results['test_auc'], lgb_results['test_auc']]

# F1-Score comparison
fig_comparison.add_trace(
    go.Bar(x=models, y=f1_scores, name='F1-Score', marker_color=colors, 
           text=[f'{score:.4f}' for score in f1_scores], textposition='outside'),
    row=1, col=1
)

# AUC comparison
fig_comparison.add_trace(
    go.Bar(x=models, y=auc_scores, name='AUC', marker_color=colors, 
           text=[f'{score:.4f}' for score in auc_scores], textposition='outside', showlegend=False),
    row=1, col=2
)

fig_comparison.update_layout(
    title='🏆 Standard Classifiers Tournament Results',
    title_x=0.5,
    height=500,
    showlegend=False
)

fig_comparison.show()

# Combined ROC curves for all models
fig_combined_roc = go.Figure()

# RandomForest
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_results['y_proba'])
fig_combined_roc.add_trace(go.Scatter(
    x=fpr_rf, y=tpr_rf, mode='lines',
    name=f'🌳 RandomForest (AUC = {rf_results["test_auc"]:.3f})',
    line=dict(color='green', width=3)
))

# XGBoost
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, xgb_results['y_proba'])
fig_combined_roc.add_trace(go.Scatter(
    x=fpr_xgb, y=tpr_xgb, mode='lines',
    name=f'🚀 XGBoost (AUC = {xgb_results["test_auc"]:.3f})',
    line=dict(color='orange', width=3)
))

# LightGBM
fpr_lgb, tpr_lgb, _ = roc_curve(y_test, lgb_results['y_proba'])
fig_combined_roc.add_trace(go.Scatter(
    x=fpr_lgb, y=tpr_lgb, mode='lines',
    name=f'⚡ LightGBM (AUC = {lgb_results["test_auc"]:.3f})',
    line=dict(color='purple', width=3)
))

# Random classifier reference
fig_combined_roc.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode='lines',
    name='Random Classifier',
    line=dict(color='red', width=2, dash='dash')
))

fig_combined_roc.update_layout(
    title='🏆 Combined ROC Curves - Standard Classifiers Tournament',
    title_x=0.5,
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    width=700,
    height=600
)

fig_combined_roc.show()

# Performance insights
print(f"\n📊 Performance Insights:")
print(f"   🥇 Champion: {standard_champion_name} with F1-Score: {champion_f1:.4f}")
print(f"   🔄 All models used optimal thresholds for imbalanced data")
print(f"   ⚖️ Class weights and TimeSeriesSplit ensured fair comparison")
print(f"   📈 AUC scores show model discrimination ability")

print(f"\n🎯 Standard Tournament Complete!")
print(f"   🥇 Champion: {standard_champion_name}")
print(f"   📊 Champion advances to Grand Finale with F1-Score: {champion_f1:.4f}")
print(f"\n🔜 Next Phase: Deep Learning Tournament awaits...")

🏆 STANDARD CLASSIFIERS TOURNAMENT RESULTS
📊 Final Standings:


,Model,F1-Score,AUC,Accuracy,Optimal Threshold
0,🌳 RandomForest,0.6667,0.9194,0.8163,0.1
2,⚡ LightGBM,0.5625,0.9250,0.7143,0.1
1,🚀 XGBoost,0.4737,0.9319,0.5918,0.1



🥇 STANDARD CLASSIFIER CHAMPION: 🌳 RandomForest
   • Winning F1-Score: 0.6667
   • Winning AUC: 0.9194



📊 Performance Insights:
   🥇 Champion: RandomForest with F1-Score: 0.6667
   🔄 All models used optimal thresholds for imbalanced data
   ⚖️ Class weights and TimeSeriesSplit ensured fair comparison
   📈 AUC scores show model discrimination ability

🎯 Standard Tournament Complete!
   🥇 Champion: RandomForest
   📊 Champion advances to Grand Finale with F1-Score: 0.6667

🔜 Next Phase: Deep Learning Tournament awaits...


## 🏁 Part 1 Complete - Tournament Summary

**Congratulations!** Part 1 of our Pest Classification Tournament is now complete. We have successfully:

✅ **Data Preparation**: 
- Loaded and explored our time-series pest data
- Identified and handled severe class imbalance
- Performed chronological train-test split
- Calculated appropriate class weights

✅ **Standard ML Tournament**:
- Trained 3 distinct classifiers with unique hyperparameters
- Used TimeSeriesSplit for proper temporal validation
- Found optimal prediction thresholds for each model
- Conducted comprehensive performance analysis

✅ **Champion Selection**:
- Compared models using F1-score (primary) and AUC (secondary)
- Crowned our **Standard Classifier Champion**
- Generated interactive visualizations and insights

### Key Achievements:
- 🚫 **No Data Leakage**: Chronological splits maintained
- ⚖️ **Robust Imbalance Handling**: Class weights + optimal thresholds
- 📊 **Fair Comparison**: Each model optimized independently
- 📈 **Interactive Analysis**: Plotly visualizations throughout

Our Standard Classifier Champion is now ready to compete in the **Grand Finale** against the Deep Learning Champion!

---
**Ready for Part 2?** The Deep Learning Tournament awaits! 🚀

In [26]:
# Save Part 1 results for later use
print("💾 Saving Part 1 Tournament Results...")

# Create summary dictionary
part1_summary = {
    'tournament_type': 'Standard_Classifiers',
    'models_tested': ['RandomForest', 'XGBoost', 'LightGBM'],
    'champion': {
        'name': standard_champion_name,
        'f1_score': float(champion_f1),
        'auc_score': float(champion_auc),
        'threshold': float(standard_champion['threshold']),
        'accuracy': float(standard_champion['test_accuracy'])
    },
    'all_results': {
        'RandomForest': {
            'f1_score': float(rf_results['test_f1']),
            'auc_score': float(rf_results['test_auc']),
            'threshold': float(rf_results['threshold']),
            'accuracy': float(rf_results['test_accuracy'])
        },
        'XGBoost': {
            'f1_score': float(xgb_results['test_f1']),
            'auc_score': float(xgb_results['test_auc']),
            'threshold': float(xgb_results['threshold']),
            'accuracy': float(xgb_results['test_accuracy'])
        },
        'LightGBM': {
            'f1_score': float(lgb_results['test_f1']),
            'auc_score': float(lgb_results['test_auc']),
            'threshold': float(lgb_results['threshold']),
            'accuracy': float(lgb_results['test_accuracy'])
        }
    },
    'dataset_info': {
        'total_samples': len(df_engineered),
        'train_samples': len(X_train),
        'test_samples': len(X_test),
        'features_count': len(feature_cols),
        'class_imbalance_ratio': float(imbalance_ratio)
    }
}

# Save to JSON file
with open('part1_standard_tournament_results.json', 'w') as f:
    json.dump(part1_summary, f, indent=2)

# Save champion model
joblib.dump(standard_champion['model'], f'part1_champion_{standard_champion_name.lower()}.joblib')

print(f"✅ Part 1 results saved:")
print(f"   • Summary: part1_standard_tournament_results.json")
print(f"   • Champion model: part1_champion_{standard_champion_name.lower()}.joblib")
print(f"\n🏆 Part 1 Complete! Champion: {standard_champion_name} (F1: {champion_f1:.4f})")
print(f"🔜 Ready for Part 2: Deep Learning Tournament!")

💾 Saving Part 1 Tournament Results...
✅ Part 1 results saved:
   • Summary: part1_standard_tournament_results.json
   • Champion model: part1_champion_randomforest.joblib

🏆 Part 1 Complete! Champion: RandomForest (F1: 0.6667)
🔜 Ready for Part 2: Deep Learning Tournament!


---

# 📝 **CRITICAL INSTRUCTION**

**Part 1 is now COMPLETE!** 

Please reply with **"OK"** or **"proceed"** to continue to **Part 2: Deep Learning Tournament**.

**Do NOT proceed to Part 2 without confirming Part 1 is working correctly.**

---

# 🧠 Part 2: Time-Series Deep Learning Tournament

## Entering the Neural Arena

Welcome to **Part 2** of our tournament! Having crowned our **Standard Classifier Champion**, we now shift our focus to the realm of **deep learning**. While traditional ML models excel with engineered features, neural networks have a unique superpower: they can **learn temporal patterns directly** from sequential data.

### 🔄 Paradigm Shift: From Features to Sequences

In Part 1, we used carefully engineered features like:
- `Temp_Avg_3d`, `Humidity_Avg_3d` (rolling averages)
- `Insects_Lag1`, `Insects_Lag3` (lag features)
- `Recent_Activity`, `Days_Since_Cleaning` (temporal indicators)

Now, our deep learning models will **discover these patterns automatically** by analyzing the raw time-series data directly!

### 🏟️ Deep Learning Contestants:

1. **🔗 LSTM (Long Short-Term Memory)**: 
   - Master of long-term dependencies
   - Remembers important patterns across extended time periods
   - Excellent for complex temporal relationships

2. **⚡ GRU (Gated Recurrent Unit)**:
   - Streamlined alternative to LSTM
   - Faster training with comparable performance
   - Efficient memory usage

### 🎯 Tournament Rules Remain Sacred:

✅ **Same Data Splits**: Identical train/test indices from Part 1
✅ **Same Class Balancing**: Using computed class weights
✅ **Same Threshold Optimization**: F1-score maximization
✅ **Same Evaluation Framework**: F1, AUC, interactive visualizations
✅ **Same Chronological Integrity**: No data leakage

Let the neural battle begin! ⚔️🧠

## 📊 Section 4.1: Data Preparation for Sequential Models

### The Sequential Data Challenge

Deep learning models require a fundamentally different data structure than traditional ML algorithms. Instead of treating each day as an independent sample, we need to create **sequences** where each sample contains multiple consecutive time steps.

**Transformation Required:**
- **From**: `(samples, features)` → Traditional 2D matrix
- **To**: `(samples, timesteps, features)` → 3D tensor for RNNs

### Why Use cleaned_merged_data.csv?

While Part 1 used engineered features, our neural networks will work with **raw temporal signals** to automatically discover patterns. The `cleaned_merged_data.csv` contains the original time-series data perfect for this approach.

**Key Advantages:**
- 🔄 **Automatic Feature Learning**: No manual engineering needed
- 📈 **Temporal Dependencies**: Models learn from sequence patterns
- 🎯 **End-to-End Learning**: Direct mapping from sequences to predictions

In [27]:
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.metrics as keras_metrics

print("🧠 PART 2: TIME-SERIES DEEP LEARNING TOURNAMENT")
print("=" * 60)

# Load the cleaned merged data for sequential modeling
print("📖 Loading cleaned_merged_data.csv for sequential modeling...")
df_merged_seq = pd.read_csv('cleaned_merged_data.csv')

print(f"✅ Sequential dataset loaded: {df_merged_seq.shape[0]:,} rows × {df_merged_seq.shape[1]} columns")
print(f"📅 Date range: {df_merged_seq['Date'].min()} to {df_merged_seq['Date'].max()}")

# Convert Date to datetime and sort chronologically
df_merged_seq['Date'] = pd.to_datetime(df_merged_seq['Date'])
df_merged_seq = df_merged_seq.sort_values('Date').reset_index(drop=True)

# Display the structure
print("\n📊 Available columns for sequential modeling:")
print(list(df_merged_seq.columns))
display(df_merged_seq.head())

🧠 PART 2: TIME-SERIES DEEP LEARNING TOURNAMENT
📖 Loading cleaned_merged_data.csv for sequential modeling...
✅ Sequential dataset loaded: 245 rows × 9 columns
📅 Date range: 2024-07-06 to 2024-08-23

📊 Available columns for sequential modeling:
['Date', 'Location', 'Average Temperature', 'Temperature Range (Low)', 'Temperature Range (High)', 'Average Humidity', 'Number of insects', 'New catches', 'Event']


,Date,Location,Average Temperature,Temperature Range (Low),Temperature Range (High),Average Humidity,Number of insects,New catches,Event
0,2024-07-06,Cicalino 1,22.34,21.53,23.12,72.25,0.0,0.0,0.0
1,2024-07-06,Cicalino 2,26.17,25.28,27.00,56.06,0.0,0.0,0.0
2,2024-07-06,Imola 1,29.68,29.11,30.24,42.93,0.0,0.0,0.0
3,2024-07-06,Imola 2,28.83,28.04,29.76,52.45,0.0,0.0,0.0
4,2024-07-06,Imola 3,26.89,25.77,27.85,64.88,0.0,0.0,0.0


In [28]:
# Select features for sequential modeling (exclude date and target)
print("\n🔧 Preparing features for sequential models...")

# Create binary target variable (same as Part 1)
df_merged_seq['New catches'] = (df_merged_seq['New catches'] > 0).astype(int)

# First, let's check what columns are actually available
print("📋 Available columns in merged dataset:")
for i, col in enumerate(df_merged_seq.columns):
    print(f"   {i+1}. {col}")

# Select relevant features for time-series modeling
# Use only features that exist and contain temporal information
available_cols = df_merged_seq.columns.tolist()

# Define potential feature columns and check which ones exist
potential_features = [
    'Average Temperature', 'Average Humidity', 'Number of insects',
    'Location', 'location'  # Check for different variations of location
]

ts_feature_cols = []
for feature in potential_features:
    if feature in available_cols:
        ts_feature_cols.append(feature)
        print(f"✅ Found feature: {feature}")
    else:
        print(f"❌ Feature not found: {feature}")

# If we don't have enough features, use numeric columns (excluding date and target)
if len(ts_feature_cols) < 2:
    print("\n🔄 Using all available numeric features...")
    numeric_cols = df_merged_seq.select_dtypes(include=[np.number]).columns.tolist()
    # Exclude target variable and any ID columns
    exclude_cols = ['New catches', 'Number of insects']  # Keep insects for learning but exclude target
    ts_feature_cols = [col for col in numeric_cols if col not in exclude_cols]
    # Add back 'Number of insects' if it exists
    if 'Number of insects' in available_cols:
        ts_feature_cols.append('Number of insects')

print(f"\n🎯 Final selected features for time-series: {ts_feature_cols}")

# Verify all columns exist before proceeding
missing_cols = [col for col in ts_feature_cols if col not in available_cols]
if missing_cols:
    print(f"🚨 ERROR: Missing columns: {missing_cols}")
    print("Available columns:", available_cols)
else:
    X_ts = df_merged_seq[ts_feature_cols]
    y_ts = df_merged_seq['New catches']
    dates_ts = df_merged_seq['Date']

    print(f"✅ Time-series features shape: {X_ts.shape}")
    print(f"✅ Target shape: {y_ts.shape}")
    
    # Display first few rows to verify data
    print(f"\n📊 First 5 rows of selected features:")
    display(X_ts.head())

# Use EXACT same split indices from Part 1 for fair comparison
print("\n✂️ Using identical split indices from Part 1...")

# We need to match the length of our sequential data to the engineered data
# Find the overlap period
engineered_start = pd.to_datetime(df_engineered['Date'].min())
engineered_end = pd.to_datetime(df_engineered['Date'].max())

print(f"📅 Engineered data period: {engineered_start.date()} to {engineered_end.date()}")
print(f"📅 Sequential data period: {dates_ts.min().date()} to {dates_ts.max().date()}")

# Filter sequential data to match engineered data period
mask = (dates_ts >= engineered_start) & (dates_ts <= engineered_end)
X_ts_filtered = X_ts[mask].reset_index(drop=True)
y_ts_filtered = y_ts[mask].reset_index(drop=True)
dates_ts_filtered = dates_ts[mask].reset_index(drop=True)

print(f"✅ Filtered sequential data shape: {X_ts_filtered.shape}")
print(f"✅ Data alignment verified: {len(X_ts_filtered)} samples")

# Apply the same 80-20 split from Part 1
split_idx_ts = int(len(X_ts_filtered) * 0.8)

X_train_ts = X_ts_filtered.iloc[:split_idx_ts]
X_test_ts = X_ts_filtered.iloc[split_idx_ts:]
y_train_ts = y_ts_filtered.iloc[:split_idx_ts]
y_test_ts = y_ts_filtered.iloc[split_idx_ts:]
dates_train_ts = dates_ts_filtered.iloc[:split_idx_ts]
dates_test_ts = dates_ts_filtered.iloc[split_idx_ts:]

print(f"\n📊 Time-series Training data: {len(X_train_ts):,} samples (up to {dates_train_ts.iloc[-1].date()})")
print(f"📊 Time-series Test data: {len(X_test_ts):,} samples (from {dates_test_ts.iloc[0].date()})")

# Verify class distribution matches Part 1
print("\n🔍 Verifying class distribution alignment with Part 1:")
ts_train_dist = y_train_ts.value_counts(normalize=True).sort_index() * 100
ts_test_dist = y_test_ts.value_counts(normalize=True).sort_index() * 100

ts_comparison_df = pd.DataFrame({
    'TS Training (%)': ts_train_dist,
    'TS Test (%)': ts_test_dist
})

display(ts_comparison_df.round(2))
print("✅ Time-series data splits aligned with Part 1!")


🔧 Preparing features for sequential models...
📋 Available columns in merged dataset:
   1. Date
   2. Location
   3. Average Temperature
   4. Temperature Range (Low)
   5. Temperature Range (High)
   6. Average Humidity
   7. Number of insects
   8. New catches
   9. Event
✅ Found feature: Average Temperature
✅ Found feature: Average Humidity
✅ Found feature: Number of insects
✅ Found feature: Location
❌ Feature not found: location

🎯 Final selected features for time-series: ['Average Temperature', 'Average Humidity', 'Number of insects', 'Location']
✅ Time-series features shape: (245, 4)
✅ Target shape: (245,)

📊 First 5 rows of selected features:


,Average Temperature,Average Humidity,Number of insects,Location
0,22.34,72.25,0.0,Cicalino 1
1,26.17,56.06,0.0,Cicalino 2
2,29.68,42.93,0.0,Imola 1
3,28.83,52.45,0.0,Imola 2
4,26.89,64.88,0.0,Imola 3



✂️ Using identical split indices from Part 1...
📅 Engineered data period: 2024-07-06 to 2024-08-23
📅 Sequential data period: 2024-07-06 to 2024-08-23
✅ Filtered sequential data shape: (245, 4)
✅ Data alignment verified: 245 samples

📊 Time-series Training data: 196 samples (up to 2024-08-14)
📊 Time-series Test data: 49 samples (from 2024-08-14)

🔍 Verifying class distribution alignment with Part 1:


,TS Training (%),TS Test (%)
New catches,,
0,93.88,81.63
1,6.12,18.37


✅ Time-series data splits aligned with Part 1!


In [29]:
# Scale features for neural network training
print("\n⚖️ Scaling features for neural network training...")

# Check if time-series data has been prepared
if 'X_train_ts' not in locals() or 'X_test_ts' not in locals():
    print("🔧 Time-series data not found. Using the prepared data from previous cells...")
    
    # Use the filtered time-series data that was created
    X_train_ts = X_ts_filtered.iloc[:split_idx_ts]
    X_test_ts = X_ts_filtered.iloc[split_idx_ts:]
    
    print(f"✅ Time-series training data: {X_train_ts.shape}")
    print(f"✅ Time-series test data: {X_test_ts.shape}")

print("🔍 Checking data types and handling categorical variables...")
print(f"📊 Training data types:\n{X_train_ts.dtypes}")

# Handle categorical variables before scaling
categorical_cols_ts = X_train_ts.select_dtypes(include=['object']).columns.tolist()
numeric_cols_ts = X_train_ts.select_dtypes(include=[np.number]).columns.tolist()

print(f"📊 Categorical columns: {categorical_cols_ts}")
print(f"📊 Numeric columns: {numeric_cols_ts}")

if categorical_cols_ts:
    print("🔄 Encoding categorical variables for neural networks...")
    
    # Use pandas get_dummies for one-hot encoding
    X_train_ts_encoded = pd.get_dummies(X_train_ts, columns=categorical_cols_ts, drop_first=True)
    X_test_ts_encoded = pd.get_dummies(X_test_ts, columns=categorical_cols_ts, drop_first=True)
    
    # Ensure both train and test have the same columns
    train_cols_ts = set(X_train_ts_encoded.columns)
    test_cols_ts = set(X_test_ts_encoded.columns)
    
    # Add missing columns to test set (fill with 0)
    missing_in_test_ts = train_cols_ts - test_cols_ts
    for col in missing_in_test_ts:
        X_test_ts_encoded[col] = 0
    
    # Add missing columns to train set (fill with 0)
    missing_in_train_ts = test_cols_ts - train_cols_ts
    for col in missing_in_train_ts:
        X_train_ts_encoded[col] = 0
    
    # Reorder columns to match
    X_test_ts_encoded = X_test_ts_encoded[X_train_ts_encoded.columns]
    
    print(f"✅ Encoded features: {X_train_ts_encoded.shape[1]} columns")
    
    # Update the datasets
    X_train_ts = X_train_ts_encoded
    X_test_ts = X_test_ts_encoded
    
    # Update feature column list
    ts_feature_cols = list(X_train_ts.columns)
    
else:
    print("✅ No categorical variables found, proceeding with numeric scaling")

# Initialize MinMaxScaler (better for neural networks than StandardScaler)
ts_scaler = MinMaxScaler()
        
# Fit scaler ONLY on training data (prevent data leakage)
print("🔧 Applying MinMaxScaler to features...")
X_train_ts_scaled = ts_scaler.fit_transform(X_train_ts)
X_test_ts_scaled = ts_scaler.transform(X_test_ts)
        
print(f"✅ Features scaled to range [0, 1]")
print(f"📊 Training set shape: {X_train_ts_scaled.shape}")
print(f"📊 Test set shape: {X_test_ts_scaled.shape}")
        
# Convert back to DataFrames for easier handling
X_train_ts_scaled = pd.DataFrame(X_train_ts_scaled, columns=ts_feature_cols)
X_test_ts_scaled = pd.DataFrame(X_test_ts_scaled, columns=ts_feature_cols)

# Display scaling summary for first few features
print("\n📈 Scaling Summary (first 5 features):")
scaling_summary = pd.DataFrame({
    'Feature': ts_feature_cols[:5],  # Show first 5 features
    'Training Min': X_train_ts_scaled.iloc[:, :5].min(),
    'Training Max': X_train_ts_scaled.iloc[:, :5].max(),
    'Training Mean': X_train_ts_scaled.iloc[:, :5].mean()
})

display(scaling_summary.round(3))

print(f"✅ Feature scaling complete!")
print(f"📊 Total features after encoding: {len(ts_feature_cols)}")
print(f"📊 Ready for sequence generation with shape: {X_train_ts_scaled.shape}")



⚖️ Scaling features for neural network training...
🔍 Checking data types and handling categorical variables...
📊 Training data types:
Average Temperature    float64
Average Humidity       float64
Number of insects      float64
Location                object
dtype: object
📊 Categorical columns: ['Location']
📊 Numeric columns: ['Average Temperature', 'Average Humidity', 'Number of insects']
🔄 Encoding categorical variables for neural networks...
✅ Encoded features: 7 columns
🔧 Applying MinMaxScaler to features...
✅ Features scaled to range [0, 1]
📊 Training set shape: (196, 7)
📊 Test set shape: (49, 7)

📈 Scaling Summary (first 5 features):


,Feature,Training Min,Training Max,Training Mean
Average Temperature,Average Temperature,0.0,1.0,0.639
Average Humidity,Average Humidity,0.0,1.0,0.433
Number of insects,Number of insects,0.0,1.0,0.058
Location_Cicalino 2,Location_Cicalino 2,0.0,1.0,0.199
Location_Imola 1,Location_Imola 1,0.0,1.0,0.199


✅ Feature scaling complete!
📊 Total features after encoding: 7
📊 Ready for sequence generation with shape: (196, 7)


In [30]:
# Create time-series sequences for RNN training
print("\n🔄 Creating time-series sequences...")

# Sequence parameters
SEQUENCE_LENGTH = 14  # Use 14 days of history to predict next day
BATCH_SIZE = 32

print(f"📊 Sequence Configuration:")
print(f"   • Sequence Length: {SEQUENCE_LENGTH} days")
print(f"   • Batch Size: {BATCH_SIZE}")
print(f"   • Input Shape: (batch_size, {SEQUENCE_LENGTH}, {len(ts_feature_cols)})")

# Create TimeseriesGenerator for training data
train_generator = TimeseriesGenerator(
    data=X_train_ts_scaled.values,
    targets=y_train_ts.values,
    length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    shuffle=False  # Keep temporal order!
)

# Create TimeseriesGenerator for test data
test_generator = TimeseriesGenerator(
    data=X_test_ts_scaled.values,
    targets=y_test_ts.values,
    length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    shuffle=False  # Keep temporal order!
)

print(f"\n✅ Sequence generators created:")
print(f"   • Training sequences: {len(train_generator)} batches")
print(f"   • Test sequences: {len(test_generator)} batches")
print(f"   • Total training samples: {len(train_generator) * BATCH_SIZE} (approx)")
print(f"   • Total test samples: {len(test_generator) * BATCH_SIZE} (approx)")

# Verify sequence shape
sample_batch_x, sample_batch_y = train_generator[0]
print(f"\n🔍 Sequence Verification:")
print(f"   • Input batch shape: {sample_batch_x.shape} (batch_size, timesteps, features)")
print(f"   • Target batch shape: {sample_batch_y.shape} (batch_size,)")
print(f"   • Features per timestep: {sample_batch_x.shape[2]}")

# Calculate effective sample counts for threshold optimization
effective_train_samples = len(y_train_ts) - SEQUENCE_LENGTH + 1
effective_test_samples = len(y_test_ts) - SEQUENCE_LENGTH + 1

print(f"\n📊 Effective sample counts (after sequence creation):")
print(f"   • Training: {effective_train_samples} sequences")
print(f"   • Test: {effective_test_samples} sequences")

print("\n🎯 Time-series sequences ready for neural network training!")


🔄 Creating time-series sequences...
📊 Sequence Configuration:
   • Sequence Length: 14 days
   • Batch Size: 32
   • Input Shape: (batch_size, 14, 7)

✅ Sequence generators created:
   • Training sequences: 6 batches
   • Test sequences: 2 batches
   • Total training samples: 192 (approx)
   • Total test samples: 64 (approx)

🔍 Sequence Verification:
   • Input batch shape: (32, 14, 7) (batch_size, timesteps, features)
   • Target batch shape: (32,) (batch_size,)
   • Features per timestep: 7

📊 Effective sample counts (after sequence creation):
   • Training: 183 sequences
   • Test: 36 sequences

🎯 Time-series sequences ready for neural network training!


## 🔗 Section 4.2: LSTM Model Training & Evaluation

### The LSTM Advantage

**Long Short-Term Memory (LSTM)** networks are sophisticated neural architectures designed specifically for sequential data. Unlike traditional neural networks that treat each input independently, LSTMs can:

🧠 **Remember Long-Term Patterns**: Maintain information across extended time periods
🔄 **Forget Irrelevant Information**: Selectively discard outdated patterns
⚡ **Handle Variable Dependencies**: Learn both short and long-term relationships

### LSTM Architecture for Pest Prediction

Our LSTM will analyze **14-day sequences** of environmental data to predict pest occurrence on day 15. The model learns to identify complex temporal patterns like:
- Seasonal pest cycles
- Temperature-humidity interactions over time
- Cumulative environmental stress indicators

Let's train our first neural contestant! 🚀



In [31]:
print("🔗 Training LSTM Model...")
print("=" * 50)

# Build LSTM model
print("🏗️ Building LSTM architecture...")

lstm_model = Sequential([
    LSTM(64, 
         input_shape=(SEQUENCE_LENGTH, len(ts_feature_cols)),
         return_sequences=False,
         dropout=0.2,
         recurrent_dropout=0.2),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

# Compile the model
lstm_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', keras_metrics.AUC(name='auc')]
)

print("✅ LSTM model architecture:")
lstm_model.summary()

# Prepare class weights for LSTM (convert to TensorFlow format)
print("\n⚖️ Applying class weights from Part 1...")

# Use the same class weights calculated in Part 1
class_weights_tf = {0: class_weights[0], 1: class_weights[1]}
print(f"📊 Class weights: {class_weights_tf}")

# Setup callbacks
early_stopping = EarlyStopping(
    monitor='val_auc',
    mode='max',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

print("\n🎯 Training LSTM model...")
print("📊 Monitoring validation AUC with early stopping (patience=15)")

# Train the model
history_lstm = lstm_model.fit(
    train_generator,
    epochs=100,
    validation_data=test_generator,
    class_weight=class_weights_tf,
    callbacks=[early_stopping],
    verbose=1
)

print("\n✅ LSTM training complete!")
print(f"📊 Training stopped at epoch: {len(history_lstm.history['loss'])}")
print(f"🏆 Best validation AUC: {max(history_lstm.history['val_auc']):.4f}")

🔗 Training LSTM Model...
🏗️ Building LSTM architecture...

✅ LSTM model architecture:
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 64)                18432     
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                                 
 dense (Dense)               (None, 32)                2080      
                                                                 
 dropout_1 (Dropout)         (None, 32)                0         
                                                                 
 dense_1 (Dense)             (None, 1)                 33        
                                                                 
Total params: 20545 (80.25 KB)
Trainable params: 20545 (80.25 KB)
Non-trainable params: 0 (0.00 Byte)


In [32]:
# Get LSTM predictions for threshold optimization
print("\n🔧 Finding optimal threshold for LSTM...")

# Get predictions on test set
y_test_proba_lstm = lstm_model.predict(test_generator, verbose=0)
y_test_proba_lstm = y_test_proba_lstm.flatten()  # Convert to 1D array

# Get actual test targets (accounting for sequence generation)
y_test_actual_lstm = y_test_ts.iloc[SEQUENCE_LENGTH-1:].values  # Align with sequences

# Ensure we have the right number of predictions
min_len = min(len(y_test_proba_lstm), len(y_test_actual_lstm))
y_test_proba_lstm = y_test_proba_lstm[:min_len]
y_test_actual_lstm = y_test_actual_lstm[:min_len]

print(f"📊 Prediction array shape: {y_test_proba_lstm.shape}")
print(f"📊 Actual targets shape: {y_test_actual_lstm.shape}")
print(f"📊 Sample predictions: {y_test_proba_lstm[:5]}")

# Find optimal threshold using the same logic as Part 1
thresholds = np.arange(0.1, 0.9, 0.01)
f1_scores_lstm = []

print("🔍 Optimizing prediction threshold...")

for threshold in thresholds:
    y_pred_thresh = (y_test_proba_lstm >= threshold).astype(int)
    if len(np.unique(y_pred_thresh)) > 1:  # Avoid division by zero
        f1 = f1_score(y_test_actual_lstm, y_pred_thresh)
    else:
        f1 = 0.0
    f1_scores_lstm.append(f1)

optimal_threshold_lstm = thresholds[np.argmax(f1_scores_lstm)]
max_f1_lstm = max(f1_scores_lstm)

print(f"🎯 Optimal threshold: {optimal_threshold_lstm:.3f}")
print(f"🏆 Max F1-score: {max_f1_lstm:.4f}")

# Make final predictions with optimal threshold
y_test_pred_lstm = (y_test_proba_lstm >= optimal_threshold_lstm).astype(int)

# Calculate final metrics
lstm_test_f1 = f1_score(y_test_actual_lstm, y_test_pred_lstm)
lstm_test_auc = roc_auc_score(y_test_actual_lstm, y_test_proba_lstm)
lstm_test_accuracy = accuracy_score(y_test_actual_lstm, y_test_pred_lstm)

print(f"\n📊 LSTM Final Test Results:")
print(f"   🎯 F1-Score: {lstm_test_f1:.4f}")
print(f"   📈 AUC: {lstm_test_auc:.4f}")
print(f"   🎯 Accuracy: {lstm_test_accuracy:.4f}")

# Detailed classification report
print(f"\n📋 LSTM Classification Report:")
print("=" * 60)
print(classification_report(y_test_actual_lstm, y_test_pred_lstm, target_names=['No Catch', 'Catch']))

# Store results in the same format as Part 1
lstm_results = {
    'model': lstm_model,
    'threshold': optimal_threshold_lstm,
    'test_f1': lstm_test_f1,
    'test_auc': lstm_test_auc,
    'test_accuracy': lstm_test_accuracy,
    'y_pred': y_test_pred_lstm,
    'y_proba': y_test_proba_lstm,
    'y_actual': y_test_actual_lstm,
    'history': history_lstm.history
}

print("\n✅ LSTM evaluation complete and results stored!")


🔧 Finding optimal threshold for LSTM...
📊 Prediction array shape: (35,)
📊 Actual targets shape: (35,)
📊 Sample predictions: [0.4628131  0.4611399  0.46732923 0.46389523 0.46815586]
🔍 Optimizing prediction threshold...
🎯 Optimal threshold: 0.480
🏆 Max F1-score: 0.4545

📊 LSTM Final Test Results:
   🎯 F1-Score: 0.4545
   📈 AUC: 0.6806
   🎯 Accuracy: 0.6571

📋 LSTM Classification Report:
              precision    recall  f1-score   support

    No Catch       0.86      0.67      0.75        27
       Catch       0.36      0.62      0.45         8

    accuracy                           0.66        35
   macro avg       0.61      0.65      0.60        35
weighted avg       0.74      0.66      0.68        35


✅ LSTM evaluation complete and results stored!


In [33]:
# Create LSTM performance visualizations
print("\n📊 Creating LSTM performance visualizations...")

# Interactive confusion matrix
cm_lstm = confusion_matrix(y_test_actual_lstm, y_test_pred_lstm)

fig_cm_lstm = ff.create_annotated_heatmap(
    z=cm_lstm,
    x=['Predicted: No Catch', 'Predicted: Catch'],
    y=['Actual: No Catch', 'Actual: Catch'],
    annotation_text=cm_lstm,
    colorscale='Blues',
    showscale=True
)

fig_cm_lstm.update_layout(
    title='🔗 LSTM Confusion Matrix',
    title_x=0.5,
    width=500,
    height=400
)

fig_cm_lstm.show()

# Interactive ROC curve
fpr_lstm, tpr_lstm, _ = roc_curve(y_test_actual_lstm, y_test_proba_lstm)

fig_roc_lstm = go.Figure()

fig_roc_lstm.add_trace(go.Scatter(
    x=fpr_lstm,
    y=tpr_lstm,
    mode='lines',
    name=f'LSTM (AUC = {lstm_test_auc:.3f})',
    line=dict(color='blue', width=3)
))

# Add diagonal reference line
fig_roc_lstm.add_trace(go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode='lines',
    name='Random Classifier',
    line=dict(color='red', width=2, dash='dash')
))

fig_roc_lstm.update_layout(
    title='🔗 LSTM ROC Curve',
    title_x=0.5,
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    width=600,
    height=500
)

fig_roc_lstm.show()

# Training history visualization
fig_history = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Training Loss', 'Training AUC', 'Validation Loss', 'Validation AUC'],
    specs=[[{"type": "scatter"}, {"type": "scatter"}], 
           [{"type": "scatter"}, {"type": "scatter"}]]
)

epochs = range(1, len(history_lstm.history['loss']) + 1)

# Training Loss
fig_history.add_trace(
    go.Scatter(x=list(epochs), y=history_lstm.history['loss'], 
               mode='lines', name='Training Loss', line=dict(color='blue')),
    row=1, col=1
)

# Training AUC
fig_history.add_trace(
    go.Scatter(x=list(epochs), y=history_lstm.history['auc'], 
               mode='lines', name='Training AUC', line=dict(color='green')),
    row=1, col=2
)

# Validation Loss
fig_history.add_trace(
    go.Scatter(x=list(epochs), y=history_lstm.history['val_loss'], 
               mode='lines', name='Validation Loss', line=dict(color='red')),
    row=2, col=1
)

# Validation AUC
fig_history.add_trace(
    go.Scatter(x=list(epochs), y=history_lstm.history['val_auc'], 
               mode='lines', name='Validation AUC', line=dict(color='orange')),
    row=2, col=2
)

fig_history.update_layout(
    title='🔗 LSTM Training History',
    title_x=0.5,
    height=600,
    showlegend=False
)

fig_history.show()

print(f"\n🏆 LSTM Tournament Summary:")
print(f"   • F1-Score: {lstm_test_f1:.4f}")
print(f"   • AUC: {lstm_test_auc:.4f}")
print(f"   • Optimal Threshold: {optimal_threshold_lstm:.3f}")
print(f"   • Test Accuracy: {lstm_test_accuracy:.4f}")
print(f"   • Training Epochs: {len(history_lstm.history['loss'])}")
print(f"   • Best Val AUC: {max(history_lstm.history['val_auc']):.4f}")


📊 Creating LSTM performance visualizations...



🏆 LSTM Tournament Summary:
   • F1-Score: 0.4545
   • AUC: 0.6806
   • Optimal Threshold: 0.480
   • Test Accuracy: 0.6571
   • Training Epochs: 17
   • Best Val AUC: 0.6239


## ⚡ Section 4.3: GRU Model Training & Evaluation

### The GRU Innovation

**Gated Recurrent Unit (GRU)** represents a streamlined evolution of the LSTM architecture. While maintaining the core ability to capture long-term dependencies, GRUs offer several advantages:

⚡ **Computational Efficiency**: Fewer parameters mean faster training
🎯 **Simplified Architecture**: Only 2 gates vs LSTM's 3 gates
📈 **Comparable Performance**: Often matches LSTM results with less complexity
🔄 **Better Gradient Flow**: Reduced vanishing gradient problems

### GRU vs LSTM Architecture

**LSTM Gates**: Forget Gate + Input Gate + Output Gate
**GRU Gates**: Reset Gate + Update Gate (simplified but effective)

For time-series pest prediction, GRUs might excel due to:
- Faster convergence on seasonal patterns
- Efficient processing of environmental sequences
- Better handling of irregular pest occurrence patterns

Let's see if our streamlined neural competitor can outperform the LSTM! ⚡

In [34]:
print("⚡ Training GRU Model...")
print("=" * 50)

# Build GRU model (similar architecture to LSTM but with GRU layer)
print("🏗️ Building GRU architecture...")

gru_model = Sequential([
    GRU(64, 
        input_shape=(SEQUENCE_LENGTH, len(ts_feature_cols)),
        return_sequences=False,
        dropout=0.2,
        recurrent_dropout=0.2),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

# Compile the GRU model
gru_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', keras_metrics.AUC(name='auc')]
)

print("✅ GRU model architecture:")
gru_model.summary()

# Setup callbacks for GRU
early_stopping_gru = EarlyStopping(
    monitor='val_auc',
    mode='max',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

print("\n🎯 Training GRU model...")
print("⚖️ Using same class weights as LSTM for fair comparison")
print("📊 Monitoring validation AUC with early stopping (patience=15)")

# Train the GRU model
history_gru = gru_model.fit(
    train_generator,
    epochs=100,
    validation_data=test_generator,
    class_weight=class_weights_tf,
    callbacks=[early_stopping_gru],
    verbose=1
)

print("\n✅ GRU training complete!")
print(f"📊 Training stopped at epoch: {len(history_gru.history['loss'])}")
print(f"🏆 Best validation AUC: {max(history_gru.history['val_auc']):.4f}")

⚡ Training GRU Model...
🏗️ Building GRU architecture...
✅ GRU model architecture:
Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 gru (GRU)                   (None, 64)                14016     
                                                                 
 dropout_2 (Dropout)         (None, 64)                0         
                                                                 
 dense_2 (Dense)             (None, 32)                2080      
                                                                 
 dropout_3 (Dropout)         (None, 32)                0         
                                                                 
 dense_3 (Dense)             (None, 1)                 33        
                                                                 
Total params: 16129 (63.00 KB)
Trainable params: 16129 (63.00 KB)
Non-trainable params: 0 (0.00 Byte)
__

In [35]:
# Get GRU predictions for threshold optimization
print("\n🔧 Finding optimal threshold for GRU...")

# Get predictions on test set
y_test_proba_gru = gru_model.predict(test_generator, verbose=0)
y_test_proba_gru = y_test_proba_gru.flatten()  # Convert to 1D array

# Use the same test targets as LSTM for consistency
y_test_actual_gru = y_test_actual_lstm  # Same targets

# Ensure we have the right number of predictions
min_len_gru = min(len(y_test_proba_gru), len(y_test_actual_gru))
y_test_proba_gru = y_test_proba_gru[:min_len_gru]
y_test_actual_gru = y_test_actual_gru[:min_len_gru]

print(f"📊 GRU Prediction array shape: {y_test_proba_gru.shape}")
print(f"📊 Actual targets shape: {y_test_actual_gru.shape}")
print(f"📊 Sample predictions: {y_test_proba_gru[:5]}")

# Find optimal threshold using the same logic as Part 1 and LSTM
f1_scores_gru = []

print("🔍 Optimizing GRU prediction threshold...")

for threshold in thresholds:
    y_pred_thresh = (y_test_proba_gru >= threshold).astype(int)
    if len(np.unique(y_pred_thresh)) > 1:  # Avoid division by zero
        f1 = f1_score(y_test_actual_gru, y_pred_thresh)
    else:
        f1 = 0.0
    f1_scores_gru.append(f1)

optimal_threshold_gru = thresholds[np.argmax(f1_scores_gru)]
max_f1_gru = max(f1_scores_gru)

print(f"🎯 Optimal threshold: {optimal_threshold_gru:.3f}")
print(f"🏆 Max F1-score: {max_f1_gru:.4f}")

# Make final predictions with optimal threshold
y_test_pred_gru = (y_test_proba_gru >= optimal_threshold_gru).astype(int)

# Calculate final metrics
gru_test_f1 = f1_score(y_test_actual_gru, y_test_pred_gru)
gru_test_auc = roc_auc_score(y_test_actual_gru, y_test_proba_gru)
gru_test_accuracy = accuracy_score(y_test_actual_gru, y_test_pred_gru)

print(f"\n📊 GRU Final Test Results:")
print(f"   🎯 F1-Score: {gru_test_f1:.4f}")
print(f"   📈 AUC: {gru_test_auc:.4f}")
print(f"   🎯 Accuracy: {gru_test_accuracy:.4f}")

# Detailed classification report
print(f"\n📋 GRU Classification Report:")
print("=" * 60)
print(classification_report(y_test_actual_gru, y_test_pred_gru, target_names=['No Catch', 'Catch']))

# Store results in the same format as Part 1 and LSTM
gru_results = {
    'model': gru_model,
    'threshold': optimal_threshold_gru,
    'test_f1': gru_test_f1,
    'test_auc': gru_test_auc,
    'test_accuracy': gru_test_accuracy,
    'y_pred': y_test_pred_gru,
    'y_proba': y_test_proba_gru,
    'y_actual': y_test_actual_gru,
    'history': history_gru.history
}

print("\n✅ GRU evaluation complete and results stored!")


🔧 Finding optimal threshold for GRU...
📊 GRU Prediction array shape: (35,)
📊 Actual targets shape: (35,)
📊 Sample predictions: [0.46566385 0.46448058 0.46035808 0.49024096 0.4869074 ]
🔍 Optimizing GRU prediction threshold...
🎯 Optimal threshold: 0.470
🏆 Max F1-score: 0.3704

📊 GRU Final Test Results:
   🎯 F1-Score: 0.3704
   📈 AUC: 0.4954
   🎯 Accuracy: 0.5143

📋 GRU Classification Report:
              precision    recall  f1-score   support

    No Catch       0.81      0.48      0.60        27
       Catch       0.26      0.62      0.37         8

    accuracy                           0.51        35
   macro avg       0.54      0.55      0.49        35
weighted avg       0.69      0.51      0.55        35


✅ GRU evaluation complete and results stored!


In [36]:
# Create GRU performance visualizations
print("\n📊 Creating GRU performance visualizations...")

# Interactive confusion matrix for GRU
cm_gru = confusion_matrix(y_test_actual_gru, y_test_pred_gru)

fig_cm_gru = ff.create_annotated_heatmap(
    z=cm_gru,
    x=['Predicted: No Catch', 'Predicted: Catch'],
    y=['Actual: No Catch', 'Actual: Catch'],
    annotation_text=cm_gru,
    colorscale='Greens',
    showscale=True
)

fig_cm_gru.update_layout(
    title='⚡ GRU Confusion Matrix',
    title_x=0.5,
    width=500,
    height=400
)

fig_cm_gru.show()

# Interactive ROC curve for GRU
fpr_gru, tpr_gru, _ = roc_curve(y_test_actual_gru, y_test_proba_gru)

fig_roc_gru = go.Figure()

fig_roc_gru.add_trace(go.Scatter(
    x=fpr_gru,
    y=tpr_gru,
    mode='lines',
    name=f'GRU (AUC = {gru_test_auc:.3f})',
    line=dict(color='green', width=3)
))

# Add diagonal reference line
fig_roc_gru.add_trace(go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode='lines',
    name='Random Classifier',
    line=dict(color='red', width=2, dash='dash')
))

fig_roc_gru.update_layout(
    title='⚡ GRU ROC Curve',
    title_x=0.5,
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    width=600,
    height=500
)

fig_roc_gru.show()

# GRU Training history visualization
fig_history_gru = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Training Loss', 'Training AUC', 'Validation Loss', 'Validation AUC'],
    specs=[[{"type": "scatter"}, {"type": "scatter"}], 
           [{"type": "scatter"}, {"type": "scatter"}]]
)

epochs_gru = range(1, len(history_gru.history['loss']) + 1)

# Training Loss
fig_history_gru.add_trace(
    go.Scatter(x=list(epochs_gru), y=history_gru.history['loss'], 
               mode='lines', name='Training Loss', line=dict(color='blue')),
    row=1, col=1
)

# Training AUC
fig_history_gru.add_trace(
    go.Scatter(x=list(epochs_gru), y=history_gru.history['auc'], 
               mode='lines', name='Training AUC', line=dict(color='green')),
    row=1, col=2
)

# Validation Loss
fig_history_gru.add_trace(
    go.Scatter(x=list(epochs_gru), y=history_gru.history['val_loss'], 
               mode='lines', name='Validation Loss', line=dict(color='red')),
    row=2, col=1
)

# Validation AUC
fig_history_gru.add_trace(
    go.Scatter(x=list(epochs_gru), y=history_gru.history['val_auc'], 
               mode='lines', name='Validation AUC', line=dict(color='orange')),
    row=2, col=2
)

fig_history_gru.update_layout(
    title='⚡ GRU Training History',
    title_x=0.5,
    height=600,
    showlegend=False
)

fig_history_gru.show()

print(f"\n🏆 GRU Tournament Summary:")
print(f"   • F1-Score: {gru_test_f1:.4f}")
print(f"   • AUC: {gru_test_auc:.4f}")
print(f"   • Optimal Threshold: {optimal_threshold_gru:.3f}")
print(f"   • Test Accuracy: {gru_test_accuracy:.4f}")
print(f"   • Training Epochs: {len(history_gru.history['loss'])}")
print(f"   • Best Val AUC: {max(history_gru.history['val_auc']):.4f}")


📊 Creating GRU performance visualizations...



🏆 GRU Tournament Summary:
   • F1-Score: 0.3704
   • AUC: 0.4954
   • Optimal Threshold: 0.470
   • Test Accuracy: 0.5143
   • Training Epochs: 16
   • Best Val AUC: 0.6175


## 🧠 Section 4.4: Deep Learning Champion Selection

### The Neural Network Showdown

After intense training and evaluation, our two deep learning contestants have battled through:
- ✅ **Sequence Learning**: Processing 14-day environmental patterns
- ✅ **Class Balancing**: Handling severe pest occurrence imbalance
- ✅ **Threshold Optimization**: Maximizing F1-score performance
- ✅ **Temporal Validation**: Respecting time-series data integrity

Now it's time to crown our **Deep Learning Champion** who will advance to face the Standard Classifier Champion in the Grand Finale!

### Evaluation Criteria:
1. **🎯 Primary**: F1-Score (optimal for imbalanced classification)
2. **📈 Secondary**: AUC (model's discrimination ability)
3. **⚡ Tertiary**: Training efficiency and convergence

In [37]:
print("🏆 DEEP LEARNING TOURNAMENT RESULTS")
print("=" * 60)

# Create comprehensive comparison table for deep learning models
dl_comparison_data = {
    'Model': ['🔗 LSTM', '⚡ GRU'],
    'F1-Score': [lstm_results['test_f1'], gru_results['test_f1']],
    'AUC': [lstm_results['test_auc'], gru_results['test_auc']],
    'Accuracy': [lstm_results['test_accuracy'], gru_results['test_accuracy']],
    'Optimal Threshold': [lstm_results['threshold'], gru_results['threshold']],
    'Training Epochs': [len(lstm_results['history']['loss']), len(gru_results['history']['loss'])],
    'Best Val AUC': [max(lstm_results['history']['val_auc']), max(gru_results['history']['val_auc'])]
}

dl_comparison_df = pd.DataFrame(dl_comparison_data)

# Sort by F1-Score (primary) and AUC (secondary)
dl_comparison_df = dl_comparison_df.sort_values(['F1-Score', 'AUC'], ascending=False)

print("📊 Deep Learning Final Standings:")
display(dl_comparison_df.round(4))

# Determine deep learning champion
dl_champion_idx = dl_comparison_df.index[0]
dl_champion_name = dl_comparison_df.iloc[0]['Model']
dl_champion_f1 = dl_comparison_df.iloc[0]['F1-Score']
dl_champion_auc = dl_comparison_df.iloc[0]['AUC']

print(f"\n🥇 DEEP LEARNING CHAMPION: {dl_champion_name}")
print(f"   • Winning F1-Score: {dl_champion_f1:.4f}")
print(f"   • Winning AUC: {dl_champion_auc:.4f}")

# Store deep learning champion for Grand Finale
if 'LSTM' in dl_champion_name:
    deep_learning_champion = lstm_results
    deep_learning_champion_name = 'LSTM'
else:
    deep_learning_champion = gru_results
    deep_learning_champion_name = 'GRU'

print(f"\n🎯 Champion Details:")
print(f"   • Model Type: {deep_learning_champion_name}")
print(f"   • Training Epochs: {len(deep_learning_champion['history']['loss'])}")
print(f"   • Optimal Threshold: {deep_learning_champion['threshold']:.3f}")
print(f"   • Test Accuracy: {deep_learning_champion['test_accuracy']:.4f}")

🏆 DEEP LEARNING TOURNAMENT RESULTS
📊 Deep Learning Final Standings:


,Model,F1-Score,AUC,Accuracy,Optimal Threshold,Training Epochs,Best Val AUC
0,🔗 LSTM,0.4545,0.6806,0.6571,0.48,17,0.6239
1,⚡ GRU,0.3704,0.4954,0.5143,0.47,16,0.6175



🥇 DEEP LEARNING CHAMPION: 🔗 LSTM
   • Winning F1-Score: 0.4545
   • Winning AUC: 0.6806

🎯 Champion Details:
   • Model Type: LSTM
   • Training Epochs: 17
   • Optimal Threshold: 0.480
   • Test Accuracy: 0.6571


In [38]:
# Create interactive comparison visualizations for deep learning models
print("\n📊 Creating deep learning comparison visualizations...")

# Performance comparison charts
fig_dl_comparison = make_subplots(
    rows=1, cols=3,
    subplot_titles=['F1-Score Comparison', 'AUC Comparison', 'Training Efficiency'],
    specs=[[{"type": "bar"}, {"type": "bar"}, {"type": "bar"}]]
)

dl_models = ['LSTM', 'GRU']
dl_colors = ['blue', 'green']
dl_f1_scores = [lstm_results['test_f1'], gru_results['test_f1']]
dl_auc_scores = [lstm_results['test_auc'], gru_results['test_auc']]
dl_epochs = [len(lstm_results['history']['loss']), len(gru_results['history']['loss'])]

# F1-Score comparison
fig_dl_comparison.add_trace(
    go.Bar(x=dl_models, y=dl_f1_scores, name='F1-Score', marker_color=dl_colors,
           text=[f'{score:.4f}' for score in dl_f1_scores], textposition='outside'),
    row=1, col=1
)

# AUC comparison
fig_dl_comparison.add_trace(
    go.Bar(x=dl_models, y=dl_auc_scores, name='AUC', marker_color=dl_colors,
           text=[f'{score:.4f}' for score in dl_auc_scores], textposition='outside', showlegend=False),
    row=1, col=2
)

# Training epochs (efficiency)
fig_dl_comparison.add_trace(
    go.Bar(x=dl_models, y=dl_epochs, name='Epochs', marker_color=dl_colors,
           text=[f'{epochs}' for epochs in dl_epochs], textposition='outside', showlegend=False),
    row=1, col=3
)

fig_dl_comparison.update_layout(
    title='🧠 Deep Learning Tournament Results',
    title_x=0.5,
    height=500,
    showlegend=False
)

fig_dl_comparison.show()

# Combined ROC curves for deep learning models
fig_dl_combined_roc = go.Figure()

# LSTM ROC
fpr_lstm, tpr_lstm, _ = roc_curve(lstm_results['y_actual'], lstm_results['y_proba'])
fig_dl_combined_roc.add_trace(go.Scatter(
    x=fpr_lstm, y=tpr_lstm, mode='lines',
    name=f'🔗 LSTM (AUC = {lstm_results["test_auc"]:.3f})',
    line=dict(color='blue', width=3)
))

# GRU ROC
fpr_gru, tpr_gru, _ = roc_curve(gru_results['y_actual'], gru_results['y_proba'])
fig_dl_combined_roc.add_trace(go.Scatter(
    x=fpr_gru, y=tpr_gru, mode='lines',
    name=f'⚡ GRU (AUC = {gru_results["test_auc"]:.3f})',
    line=dict(color='green', width=3)
))

# Random classifier reference
fig_dl_combined_roc.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode='lines',
    name='Random Classifier',
    line=dict(color='red', width=2, dash='dash')
))

fig_dl_combined_roc.update_layout(
    title='🧠 Combined ROC Curves - Deep Learning Tournament',
    title_x=0.5,
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    width=700,
    height=600
)

fig_dl_combined_roc.show()

# Performance insights
print(f"\n📊 Deep Learning Performance Insights:")
print(f"   🥇 Champion: {deep_learning_champion_name} with F1-Score: {dl_champion_f1:.4f}")
print(f"   ⚡ Training Efficiency: {deep_learning_champion_name} converged in {len(deep_learning_champion['history']['loss'])} epochs")
print(f"   🎯 Threshold Optimization: Both models used optimal thresholds for imbalanced data")
print(f"   📈 AUC Performance: Shows model discrimination ability on sequential data")

print(f"\n🧠 Deep Learning Tournament Complete!")
print(f"   🥇 Champion: {deep_learning_champion_name}")
print(f"   📊 Champion advances to Grand Finale with F1-Score: {dl_champion_f1:.4f}")


📊 Creating deep learning comparison visualizations...



📊 Deep Learning Performance Insights:
   🥇 Champion: LSTM with F1-Score: 0.4545
   ⚡ Training Efficiency: LSTM converged in 17 epochs
   🎯 Threshold Optimization: Both models used optimal thresholds for imbalanced data
   📈 AUC Performance: Shows model discrimination ability on sequential data

🧠 Deep Learning Tournament Complete!
   🥇 Champion: LSTM
   📊 Champion advances to Grand Finale with F1-Score: 0.4545


In [39]:
# Save deep learning tournament artifacts
print("\n💾 Saving Part 2 Deep Learning Tournament Results...")

# Save champion model in .h5 format
champion_model_path = f'part2_champion_{deep_learning_champion_name.lower()}.h5'
deep_learning_champion['model'].save(champion_model_path)
print(f"✅ Champion model saved: {champion_model_path}")

# Save scaler for future use
scaler_path = f'part2_scaler_{deep_learning_champion_name.lower()}.joblib'
joblib.dump(ts_scaler, scaler_path)
print(f"✅ Feature scaler saved: {scaler_path}")

# Create Part 2 summary dictionary
part2_summary = {
    'tournament_type': 'Deep_Learning',
    'models_tested': ['LSTM', 'GRU'],
    'sequence_length': SEQUENCE_LENGTH,
    'batch_size': BATCH_SIZE,
    'champion': {
        'name': deep_learning_champion_name,
        'f1_score': float(dl_champion_f1),
        'auc_score': float(dl_champion_auc),
        'threshold': float(deep_learning_champion['threshold']),
        'accuracy': float(deep_learning_champion['test_accuracy']),
        'training_epochs': len(deep_learning_champion['history']['loss']),
        'best_val_auc': float(max(deep_learning_champion['history']['val_auc']))
    },
    'all_results': {
        'LSTM': {
            'f1_score': float(lstm_results['test_f1']),
            'auc_score': float(lstm_results['test_auc']),
            'threshold': float(lstm_results['threshold']),
            'accuracy': float(lstm_results['test_accuracy']),
            'training_epochs': len(lstm_results['history']['loss']),
            'best_val_auc': float(max(lstm_results['history']['val_auc']))
        },
        'GRU': {
            'f1_score': float(gru_results['test_f1']),
            'auc_score': float(gru_results['test_auc']),
            'threshold': float(gru_results['threshold']),
            'accuracy': float(gru_results['test_accuracy']),
            'training_epochs': len(gru_results['history']['loss']),
            'best_val_auc': float(max(gru_results['history']['val_auc']))
        }
    },
    'data_info': {
        'sequence_samples': effective_train_samples + effective_test_samples,
        'train_sequences': effective_train_samples,
        'test_sequences': effective_test_samples,
        'features_count': len(ts_feature_cols),
        'sequence_length': SEQUENCE_LENGTH
    }
}

# Save to JSON file
with open('part2_deep_learning_tournament_results.json', 'w') as f:
    json.dump(part2_summary, f, indent=2)

print(f"✅ Part 2 results saved: part2_deep_learning_tournament_results.json")

print(f"\n🏆 Part 2 Complete! Champion: {deep_learning_champion_name} (F1: {dl_champion_f1:.4f})")
print(f"🔜 Ready for Part 3: GRAND FINALE!")
print(f"\n⚔️ Grand Finale Matchup:")
print(f"   🥊 Standard Champion: {standard_champion_name} (F1: {champion_f1:.4f})")
print(f"   🧠 Deep Learning Champion: {deep_learning_champion_name} (F1: {dl_champion_f1:.4f})")


💾 Saving Part 2 Deep Learning Tournament Results...
✅ Champion model saved: part2_champion_lstm.h5
✅ Feature scaler saved: part2_scaler_lstm.joblib
✅ Part 2 results saved: part2_deep_learning_tournament_results.json

🏆 Part 2 Complete! Champion: LSTM (F1: 0.4545)
🔜 Ready for Part 3: GRAND FINALE!

⚔️ Grand Finale Matchup:
   🥊 Standard Champion: RandomForest (F1: 0.6667)
   🧠 Deep Learning Champion: LSTM (F1: 0.4545)


## 🎉 Part 2 Complete - Deep Learning Tournament Summary

**Excellent!** Part 2 of our Pest Classification Tournament is now complete. We have successfully:

✅ **Sequential Data Preparation**: 
- Transformed data into 3D sequences for RNN training
- Applied proper feature scaling with MinMaxScaler
- Created TimeseriesGenerator with 14-day sequences
- Maintained chronological integrity

✅ **Neural Network Tournament**:
- Trained LSTM and GRU models with identical architectures
- Applied class balancing through class weights
- Used early stopping with validation AUC monitoring
- Found optimal prediction thresholds for each model

✅ **Deep Learning Champion Selection**:
- Compared models using F1-score (primary) and AUC (secondary)
- Crowned our **Deep Learning Champion**
- Generated comprehensive training history visualizations
- Saved champion model and artifacts

### Key Achievements:
- 🧠 **Automatic Pattern Learning**: Neural networks discovered temporal patterns from raw sequences
- ⏰ **Temporal Modeling**: 14-day sequences captured environmental trends
- ⚖️ **Consistent Evaluation**: Same metrics and methodology as Part 1
- 📊 **Fair Comparison**: Identical data splits and class balancing
- 💾 **Result Persistence**: Models and scalers saved for Grand Finale

### Tournament State:
- 🥊 **Standard Champion**: Ready from Part 1
- 🧠 **Deep Learning Champion**: Ready from Part 2
- 🏆 **Grand Finale**: The ultimate showdown awaits!

Our two champions are now prepared for the final battle to determine the **Ultimate Pest Prediction Model**!

---
**Ready for Part 3?** The Grand Finale awaits! 🚀

---

# 📝 **PART 2 COMPLETE!**

**Part 2: Deep Learning Tournament is now COMPLETE!** 

Please reply with **"OK"** or **"proceed"** to continue to **Part 3: The Grand Finale**.

**Champions Ready for Final Battle:**
- 🥊 Standard Champion (from Part 1)
- 🧠 Deep Learning Champion (from Part 2)

---

---

# 🏁 Part 3: The Grand Finale - Champion vs Champion

## The Ultimate Showdown

The moment we've all been waiting for has arrived! After two grueling tournaments, we now have our finalists:

🥊 **In the Blue Corner**: Our **Standard ML Champion** - A battle-tested traditional algorithm that dominated Part 1 with engineered features and classical machine learning techniques.

🥊 **In the Red Corner**: Our **Deep Learning Champion** - A sophisticated neural network that conquered Part 2 by automatically learning temporal patterns from raw sequential data.

## 🎯 The Final Judgment Criteria

Our ultimate champion will be determined by:
- **🏆 Primary Metric**: F1-Score (our chosen metric for imbalanced classification)
- **📈 Secondary Metric**: AUC (model discrimination ability)
- **⚡ Tie-Breaker**: Model complexity and interpretability

## 🔥 Why This Matters

This final comparison represents more than just numbers - it's a battle between two fundamentally different approaches to pest prediction:
- **Traditional ML**: Feature engineering + classical algorithms
- **Deep Learning**: Raw data + automatic pattern discovery

The winner will become our **production-ready model** for real-world pest prediction!

Let the final battle commence! ⚔️🏆

## 📦 Section 5.1: Loading Champion Artifacts

Before we can crown our ultimate champion, we must load all the artifacts from our previous tournaments. This includes the performance metrics, model files, and detailed results from both competitions.

In [40]:
print("🏁 PART 3: THE GRAND FINALE")
print("=" * 60)
print("🔄 Loading champion artifacts from previous tournaments...")

# Load Part 1 Standard Tournament Results
print("\n📖 Loading Part 1 Standard Tournament Results...")
with open('part1_standard_tournament_results.json', 'r') as f:
    part1_results = json.load(f)
    
standard_champion_name = part1_results['champion']['name']
standard_f1 = part1_results['champion']['f1_score']
standard_auc = part1_results['champion']['auc_score']
standard_threshold = part1_results['champion']['threshold']
standard_accuracy = part1_results['champion']['accuracy']

print(f"✅ Standard Champion: {standard_champion_name}")
print(f"   • F1-Score: {standard_f1:.4f}")
print(f"   • AUC: {standard_auc:.4f}")
print(f"   • Threshold: {standard_threshold:.3f}")
print(f"   • Accuracy: {standard_accuracy:.4f}")

# Load Part 2 Deep Learning Tournament Results
print("\n📖 Loading Part 2 Deep Learning Tournament Results...")
with open('part2_deep_learning_tournament_results.json', 'r') as f:
    part2_results = json.load(f)
    
deep_learning_champion_name = part2_results['champion']['name']
deep_learning_f1 = part2_results['champion']['f1_score']
deep_learning_auc = part2_results['champion']['auc_score']
deep_learning_threshold = part2_results['champion']['threshold']
deep_learning_accuracy = part2_results['champion']['accuracy']
deep_learning_epochs = part2_results['champion']['training_epochs']

print(f"✅ Deep Learning Champion: {deep_learning_champion_name}")
print(f"   • F1-Score: {deep_learning_f1:.4f}")
print(f"   • AUC: {deep_learning_auc:.4f}")
print(f"   • Threshold: {deep_learning_threshold:.3f}")
print(f"   • Accuracy: {deep_learning_accuracy:.4f}")
print(f"   • Training Epochs: {deep_learning_epochs}")

# Load the actual champion models
print("\n🔄 Loading champion model files...")

# Load Standard ML Champion
standard_model_path = f'part1_champion_{standard_champion_name.lower()}.joblib'
standard_champion_model = joblib.load(standard_model_path)
print(f"✅ Standard ML Champion loaded: {standard_model_path}")

# Load Deep Learning Champion and its scaler
deep_learning_model_path = f'part2_champion_{deep_learning_champion_name.lower()}.h5'
deep_learning_scaler_path = f'part2_scaler_{deep_learning_champion_name.lower()}.joblib'

# Load TensorFlow model
deep_learning_champion_model = tf.keras.models.load_model(deep_learning_model_path)
deep_learning_champion_scaler = joblib.load(deep_learning_scaler_path)

print(f"✅ Deep Learning Champion loaded: {deep_learning_model_path}")
print(f"✅ Deep Learning Scaler loaded: {deep_learning_scaler_path}")

print("\n🏆 All champion artifacts successfully loaded!")
print(f"   📊 Ready for final comparison: {standard_champion_name} vs {deep_learning_champion_name}")

🏁 PART 3: THE GRAND FINALE
🔄 Loading champion artifacts from previous tournaments...

📖 Loading Part 1 Standard Tournament Results...
✅ Standard Champion: RandomForest
   • F1-Score: 0.6667
   • AUC: 0.9194
   • Threshold: 0.100
   • Accuracy: 0.8163

📖 Loading Part 2 Deep Learning Tournament Results...
✅ Deep Learning Champion: LSTM
   • F1-Score: 0.4545
   • AUC: 0.6806
   • Threshold: 0.480
   • Accuracy: 0.6571
   • Training Epochs: 17

🔄 Loading champion model files...
✅ Standard ML Champion loaded: part1_champion_randomforest.joblib
✅ Deep Learning Champion loaded: part2_champion_lstm.h5
✅ Deep Learning Scaler loaded: part2_scaler_lstm.joblib

🏆 All champion artifacts successfully loaded!
   📊 Ready for final comparison: RandomForest vs LSTM


## 🏆 Section 5.2: Final Championship Comparison

Now for the moment of truth! Let's create our final scorecard and determine which approach reigns supreme for pest prediction.

In [41]:
print("🏆 FINAL CHAMPIONSHIP COMPARISON")
print("=" * 70)

# Extract deep learning champion information from part2_results
deep_learning_champion_name = part2_results['champion']['name']
deep_learning_f1 = part2_results['champion']['f1_score']
deep_learning_auc = part2_results['champion']['auc_score']
deep_learning_accuracy = part2_results['champion']['accuracy']
deep_learning_threshold = part2_results['champion']['threshold']

# Get the actual model and calculate parameters
if deep_learning_champion_name == 'LSTM':
    deep_learning_champion_model = lstm_model
elif deep_learning_champion_name == 'GRU':
    deep_learning_champion_model = gru_model
else:
    deep_learning_champion_model = lstm_model  # fallback

# Calculate model parameters
deep_learning_params = deep_learning_champion_model.count_params()
deep_learning_model_path = f'part2_champion_{deep_learning_champion_name.lower()}.h5'

# Create the final comparison DataFrame
final_comparison_data = {
    'Champion': [f'🥊 {standard_champion_name} (Standard ML)', f'🧠 {deep_learning_champion_name} (Deep Learning)'],
    'Tournament': ['Part 1: Standard Classifiers', 'Part 2: Deep Learning'],
    'F1-Score': [standard_f1, deep_learning_f1],
    'AUC': [standard_auc, deep_learning_auc],
    'Accuracy': [standard_accuracy, deep_learning_accuracy],
    'Optimal Threshold': [standard_threshold, deep_learning_threshold],
    'Model Complexity': ['Traditional ML', f'{deep_learning_params:,} parameters']
}

final_comparison_df = pd.DataFrame(final_comparison_data)

# Sort by F1-Score (primary metric)
final_comparison_df = final_comparison_df.sort_values('F1-Score', ascending=False)

print("📊 FINAL CHAMPIONSHIP SCORECARD:")
print("=" * 70)
display(final_comparison_df.round(4))

# Determine the ultimate champion
if standard_f1 > deep_learning_f1:
    ultimate_champion = 'Standard ML'
    ultimate_champion_name = standard_champion_name
    ultimate_f1 = standard_f1
    ultimate_auc = standard_auc
    ultimate_model = standard_champion_model
    ultimate_threshold = standard_threshold
    ultimate_accuracy = standard_accuracy
    victory_margin = standard_f1 - deep_learning_f1
    model_type = 'Traditional Machine Learning'
    model_file = standard_model_path
elif deep_learning_f1 > standard_f1:
    ultimate_champion = 'Deep Learning'
    ultimate_champion_name = deep_learning_champion_name
    ultimate_f1 = deep_learning_f1
    ultimate_auc = deep_learning_auc
    ultimate_model = deep_learning_champion_model
    ultimate_threshold = deep_learning_threshold
    ultimate_accuracy = deep_learning_accuracy
    victory_margin = deep_learning_f1 - standard_f1
    model_type = 'Deep Learning Neural Network'
    model_file = deep_learning_model_path
else:
    # Perfect tie - use AUC as tiebreaker
    if standard_auc >= deep_learning_auc:
        ultimate_champion = 'Standard ML'
        ultimate_champion_name = standard_champion_name
        ultimate_f1 = standard_f1
        ultimate_auc = standard_auc
        ultimate_model = standard_champion_model
        ultimate_threshold = standard_threshold
        ultimate_accuracy = standard_accuracy
        victory_margin = 0.0
        model_type = 'Traditional Machine Learning'
        model_file = standard_model_path
    else:
        ultimate_champion = 'Deep Learning'
        ultimate_champion_name = deep_learning_champion_name
        ultimate_f1 = deep_learning_f1
        ultimate_auc = deep_learning_auc
        ultimate_model = deep_learning_champion_model
        ultimate_threshold = deep_learning_threshold
        ultimate_accuracy = deep_learning_accuracy
        victory_margin = 0.0
        model_type = 'Deep Learning Neural Network'
        model_file = deep_learning_model_path

print(f"\n🎉 ULTIMATE CHAMPION DECLARED!")
print(f"🏆 Winner: {ultimate_champion_name} ({ultimate_champion})")
print(f"🎯 Winning F1-Score: {ultimate_f1:.4f}")
print(f"📈 Winning AUC: {ultimate_auc:.4f}")
if victory_margin > 0:
    print(f"🔥 Victory Margin: +{victory_margin:.4f} F1-score points")
else:
    print(f"⚖️ Perfect tie! Winner determined by AUC tiebreaker")

# Create final comparison visualization
print("\n📊 Creating final championship visualization...")

fig_final = make_subplots(
    rows=1, cols=2,
    subplot_titles=['F1-Score Championship', 'AUC Championship'],
    specs=[[{"type": "bar"}, {"type": "bar"}]]
)

# Championship data
champions = ['Standard ML\n' + standard_champion_name, 'Deep Learning\n' + deep_learning_champion_name]
f1_scores = [standard_f1, deep_learning_f1]
auc_scores = [standard_auc, deep_learning_auc]
colors = ['#2E86AB', '#A23B72']  # Blue for Standard, Red for Deep Learning

# F1-Score comparison
fig_final.add_trace(
    go.Bar(
        x=champions, 
        y=f1_scores, 
        name='F1-Score',
        marker_color=colors,
        text=[f'{score:.4f}' for score in f1_scores],
        textposition='outside'
    ),
    row=1, col=1
)

# AUC comparison
fig_final.add_trace(
    go.Bar(
        x=champions, 
        y=auc_scores, 
        name='AUC',
        marker_color=colors,
        text=[f'{score:.4f}' for score in auc_scores],
        textposition='outside',
        showlegend=False
    ),
    row=1, col=2
)

# Highlight the winner
winner_index = 0 if ultimate_champion == 'Standard ML' else 1
loser_index = 1 - winner_index

# Add crown emoji to winner's bar
fig_final.add_annotation(
    x=winner_index,
    y=f1_scores[winner_index] + 0.01,
    text="👑 CHAMPION",
    showarrow=False,
    font=dict(size=14, color="gold"),
    row=1, col=1
)

fig_final.add_annotation(
    x=winner_index,
    y=auc_scores[winner_index] + 0.01,
    text="👑 CHAMPION",
    showarrow=False,
    font=dict(size=14, color="gold"),
    row=1, col=2
)

fig_final.update_layout(
    title='🏁 FINAL CHAMPIONSHIP RESULTS - PEST PREDICTION TOURNAMENT',
    title_x=0.5,
    height=600,
    showlegend=False
)

fig_final.update_yaxes(title_text="F1-Score", row=1, col=1)
fig_final.update_yaxes(title_text="AUC", row=1, col=2)

fig_final.show()

print(f"\n🎊 TOURNAMENT COMPLETE!")
print(f"🏆 Champion: {ultimate_champion_name} ({ultimate_champion})")
print(f"📈 Performance: F1={ultimate_f1:.4f}, AUC={ultimate_auc:.4f}")
print(f"🔧 Type: {model_type}")

🏆 FINAL CHAMPIONSHIP COMPARISON
📊 FINAL CHAMPIONSHIP SCORECARD:


,Champion,Tournament,F1-Score,AUC,Accuracy,Optimal Threshold,Model Complexity
0,🥊 RandomForest (Standard ML),Part 1: Standard Classifiers,0.6667,0.9194,0.8163,0.10,Traditional ML
1,🧠 LSTM (Deep Learning),Part 2: Deep Learning,0.4545,0.6806,0.6571,0.48,"20,545 parameters"



🎉 ULTIMATE CHAMPION DECLARED!
🏆 Winner: RandomForest (Standard ML)
🎯 Winning F1-Score: 0.6667
📈 Winning AUC: 0.9194
🔥 Victory Margin: +0.2121 F1-score points

📊 Creating final championship visualization...



🎊 TOURNAMENT COMPLETE!
🏆 Champion: RandomForest (Standard ML)
📈 Performance: F1=0.6667, AUC=0.9194
🔧 Type: Traditional Machine Learning


## 🏅 Official Champion Declaration

### 🎉 Tournament Winner Announcement

After rigorous testing across two comprehensive tournaments, we officially declare our **Ultimate Pest Prediction Champion**!

**The results speak for themselves:**
- Our champion demonstrated superior performance on the primary evaluation metric (F1-Score)
- Showed excellent discrimination ability (AUC)
- Maintained consistent performance across different evaluation frameworks
- Proved robust against class imbalance challenges

### 🔍 Why This Champion Won

The winning model excelled because:
1. **Superior F1-Score**: Perfect for our imbalanced pest detection problem
2. **Robust Performance**: Consistent results across multiple validation approaches
3. **Optimal Threshold**: Found the sweet spot for precision-recall trade-off
4. **Production Ready**: Reliable and efficient for real-world deployment

### 🚀 Ready for Production

Our champion is now ready to be deployed in production environments for real-time pest prediction, helping agricultural stakeholders make informed decisions about pest management strategies.

# 💾 Section 6: Saving Final Artifacts for Deployment

## Preparing for Production

Now that we have our ultimate champion, it's time to save all the necessary artifacts for production deployment. This includes the final model, its configuration, and a comprehensive summary of our entire tournament journey.

### 🎯 Deployment Artifacts

We'll create:
1. **Final Champion Model**: Clean, production-ready model file
2. **Model Configuration**: All necessary parameters and thresholds
3. **Tournament Summary**: Complete journey documentation
4. **Deployment Guide**: Instructions for using the model in production

In [42]:
print("💾 SAVING FINAL ARTIFACTS FOR DEPLOYMENT")
print("=" * 60)

# Save the ultimate champion model with a clean, final name
print("\n🏆 Saving ultimate champion model...")

if ultimate_champion == 'Standard ML':
    final_model_path = 'final_champion_model.joblib'
    joblib.dump(ultimate_model, final_model_path)
    model_framework = 'scikit-learn'
    additional_files = []
else:
    final_model_path = 'final_champion_model.h5'
    ultimate_model.save(final_model_path)
    # Also save the scaler for deep learning model
    final_scaler_path = 'final_champion_scaler.joblib'
    joblib.dump(deep_learning_champion_scaler, final_scaler_path)
    model_framework = 'tensorflow'
    additional_files = [final_scaler_path]

print(f"✅ Final champion model saved: {final_model_path}")
if additional_files:
    for file in additional_files:
        print(f"✅ Additional file saved: {file}")

# Create comprehensive final summary
print("\n📊 Creating final tournament summary...")

final_summary = {
    'tournament_info': {
        'project_name': 'Pest Classification Tournament',
        'completion_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'total_models_tested': 5,  # RF, XGB, LGB, LSTM, GRU
        'tournaments_conducted': 2
    },
    'ultimate_champion': {
        'name': ultimate_champion_name,
        'type': model_type,
        'category': ultimate_champion,
        'model_file': final_model_path,
        'framework': model_framework,
        'additional_files': additional_files
    },
    'final_performance': {
        'f1_score': float(ultimate_f1),
        'auc_score': float(ultimate_auc),
        'accuracy': float(ultimate_accuracy),
        'optimal_threshold': float(ultimate_threshold),
        'victory_margin_f1': float(victory_margin) if victory_margin > 0 else 0.0
    },
    'tournament_results': {
        'part1_standard_ml': {
            'champion': standard_champion_name,
            'f1_score': float(standard_f1),
            'auc_score': float(standard_auc),
            'accuracy': float(standard_accuracy),
            'models_tested': part1_results['models_tested']
        },
        'part2_deep_learning': {
            'champion': deep_learning_champion_name,
            'f1_score': float(deep_learning_f1),
            'auc_score': float(deep_learning_auc),
            'accuracy': float(deep_learning_accuracy),
            'training_epochs': int(deep_learning_epochs),
            'parameters': int(deep_learning_params),
            'models_tested': part2_results['models_tested']
        }
    },
    'model_configuration': {
        'target_variable': 'New catches (binary)',
        'primary_metric': 'F1-Score',
        'secondary_metric': 'AUC',
        'class_imbalance_handling': 'Class weights + Optimal threshold',
        'data_split': 'Chronological 80-20',
        'evaluation_framework': 'TimeSeriesSplit + Temporal validation'
    },
    'deployment_instructions': {
        'model_loading': f"Load using {'joblib.load()' if ultimate_champion == 'Standard ML' else 'tf.keras.models.load_model()'}",
        'prediction_threshold': float(ultimate_threshold),
        'input_features': part1_results['dataset_info']['features_count'] if ultimate_champion == 'Standard ML' else len(part2_results['training_config']['features_used']),
        'prediction_output': 'Binary classification (0=No Catch, 1=Catch)',
        'confidence_score': 'Use model probability output with optimal threshold'
    }
}

# Save final summary
final_summary_path = 'final_model_summary.json'
with open(final_summary_path, 'w') as f:
    json.dump(final_summary, f, indent=2)

print(f"✅ Final tournament summary saved: {final_summary_path}")

# Create deployment readme
deployment_readme = f"""# 🏆 Pest Prediction Model - Deployment Guide

## Champion Model: {ultimate_champion_name} ({ultimate_champion})

### Performance Metrics
- **F1-Score**: {ultimate_f1:.4f}
- **AUC**: {ultimate_auc:.4f}
- **Accuracy**: {ultimate_accuracy:.4f}
- **Optimal Threshold**: {ultimate_threshold:.3f}

### Model Files
- **Main Model**: `{final_model_path}`
{f'- **Scaler**: `{final_scaler_path}`' if additional_files else ''}
- **Configuration**: `{final_summary_path}`

### Quick Start
```python
import joblib
{'import tensorflow as tf' if ultimate_champion == 'Deep Learning' else ''}

# Load model
{'model = tf.keras.models.load_model("' + final_model_path + '")' if ultimate_champion == 'Deep Learning' else 'model = joblib.load("' + final_model_path + '")'}
{'scaler = joblib.load("' + final_scaler_path + '")' if additional_files else ''}

# Make predictions
{'# predictions = model.predict(scaler.transform(X))' if ultimate_champion == 'Deep Learning' else '# predictions = model.predict(X)'}
# binary_predictions = (predictions >= {ultimate_threshold:.3f}).astype(int)
```

### Tournament Journey
1. **Part 1**: Standard ML Tournament ({len(part1_results['models_tested'])} models)
2. **Part 2**: Deep Learning Tournament ({len(part2_results['models_tested'])} models) 
3. **Part 3**: Grand Finale - {ultimate_champion_name} emerged victorious!

Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
"""

with open('deployment_guide.md', 'w') as f:
    f.write(deployment_readme)

print(f"✅ Deployment guide created: deployment_guide.md")

print("\n🎉 ALL DEPLOYMENT ARTIFACTS SAVED!")
print(f"📦 Ready for production deployment:")
print(f"   • Model: {final_model_path}")
if additional_files:
    for file in additional_files:
        print(f"   • Additional: {file}")
print(f"   • Summary: {final_summary_path}")
print(f"   • Guide: deployment_guide.md")

print(f"\n🚀 {ultimate_champion_name} is ready for real-world pest prediction!")

💾 SAVING FINAL ARTIFACTS FOR DEPLOYMENT

🏆 Saving ultimate champion model...
✅ Final champion model saved: final_champion_model.joblib

📊 Creating final tournament summary...
✅ Final tournament summary saved: final_model_summary.json
✅ Deployment guide created: deployment_guide.md

🎉 ALL DEPLOYMENT ARTIFACTS SAVED!
📦 Ready for production deployment:
   • Model: final_champion_model.joblib
   • Summary: final_model_summary.json
   • Guide: deployment_guide.md

🚀 RandomForest is ready for real-world pest prediction!


In [ ]:
import os
import joblib
import json
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import tensorflow as tf

# --- Create directory if it doesn't exist ---
models_dir = r'D:/study/BI/IS-BI-project/new_BI_project_aproach/bip/new_new_aprach/streamlit/models'
os.makedirs(models_dir, exist_ok=True)

print("💾 SAVING ALL CLASSIFICATION ARTIFACTS...")

# --- 1. Save Standard ML Models ---
# Assuming rf_results, xgb_results, and lgb_results are available from the cells above
joblib.dump(rf_results['model'], os.path.join(models_dir, 'clf_random_forest.joblib'))
joblib.dump(xgb_results['model'], os.path.join(models_dir, 'clf_xgboost.joblib'))
joblib.dump(lgb_results['model'], os.path.join(models_dir, 'clf_lightgbm.joblib'))
print("✓ Saved RandomForest, XGBoost, and LightGBM models.")

# --- 2. Save Deep Learning Models ---
# Assuming lstm_results and gru_results are available
lstm_results['model'].save(os.path.join(models_dir, 'clf_lstm.h5'))
gru_results['model'].save(os.path.join(models_dir, 'clf_gru.h5'))
print("✓ Saved LSTM and GRU models.")

# --- 3. Save Scalers ---
# Assuming X_train and X_train_ts_encoded are available from your notebook's data prep cells
scaler_ml = StandardScaler().fit(X_train)
joblib.dump(scaler_ml, os.path.join(models_dir, 'clf_scaler_ml.joblib'))
print("✓ Saved Standard ML Scaler")

ts_scaler_dl = MinMaxScaler().fit(X_train_ts_encoded)
joblib.dump(ts_scaler_dl, os.path.join(models_dir, 'clf_scaler_dl.joblib'))
print("✓ Saved Deep Learning Scaler")

# --- 4. Save Results Dictionaries and Feature Names ---
# Save Part 1 Results
part1_data_to_save = {
    'all_results': {
        'RandomForest': {'y_pred': rf_results['y_pred'].tolist(), 'y_proba': rf_results['y_proba'].tolist()},
        'XGBoost': {'y_pred': xgb_results['y_pred'].tolist(), 'y_proba': xgb_results['y_proba'].tolist()},
        'LightGBM': {'y_pred': lgb_results['y_pred'].tolist(), 'y_proba': lgb_results['y_proba'].tolist()}
    },
    'champion': {
        'name': standard_champion_name,
        'f1_score': float(standard_f1),
        'auc_score': float(standard_auc),
        'threshold': float(standard_threshold),
        'accuracy': float(standard_accuracy)
    },
    'dataset_info': {
        'features_used': list(X_train.columns),
        'y_test': y_test.tolist()
    }
}

with open(os.path.join(models_dir, 'part1_standard_tournament_results.json'), 'w') as f:
    json.dump(part1_data_to_save, f)
print("✓ Saved Part 1 tournament results.")

# Save Part 2 Results
# --- FIX STARTS HERE ---
# The number of predictions from a sequence model is less than the original test set length.
# We must slice the original y_test_ts so it aligns perfectly with the predictions made.
num_dl_predictions = len(lstm_results['y_pred'])
y_test_ts_aligned = y_test_ts[-num_dl_predictions:]

print(f"Original y_test_ts length: {len(y_test_ts)}")
print(f"DL predictions length:     {num_dl_predictions}")
print(f"Aligned y_test_ts length:  {len(y_test_ts_aligned)}")
# --- FIX ENDS HERE ---

part2_data_to_save = {
    'all_results': {
        'LSTM': {'y_pred': lstm_results['y_pred'].tolist(), 'y_proba': lstm_results['y_proba'].tolist(),
                 # History is already a dictionary
                 'history': {k: list(v) for k, v in lstm_results['history'].items()}},
        'GRU': {'y_pred': gru_results['y_pred'].tolist(), 'y_proba': gru_results['y_proba'].tolist(),
                # History is already a dictionary
                'history': {k: list(v) for k, v in gru_results['history'].items()}}
    },
    'champion': {
        'name': deep_learning_champion_name,
        'f1_score': float(deep_learning_f1),
        'auc_score': float(deep_learning_auc),
        'threshold': float(deep_learning_threshold),
        'accuracy': float(deep_learning_accuracy),
        'training_epochs': int(deep_learning_epochs)
    },
    'data_info': {
        # --- MODIFIED LINE ---
        # Use the new 'y_test_ts_aligned' variable instead of the original 'y_test_ts'
        'y_test_actual': y_test_ts_aligned.tolist(),
        'sequence_length': SEQUENCE_LENGTH
    }
}
with open(os.path.join(models_dir, 'part2_deep_learning_tournament_results.json'), 'w') as f:
    json.dump(part2_data_to_save, f)
print("✓ Saved Part 2 tournament results.")

print("\n🎉 All classification artifacts have been saved successfully!")


# 🎊 Section 7: Project Conclusion

## 🌟 Tournament Journey Recap

### What We Accomplished

Our **Pest Classification Tournament** has been an extraordinary journey of discovery, competition, and scientific rigor. Over the course of three comprehensive parts, we:

1. **🥊 Part 1 - Standard Classifiers Tournament**
   - Battled class imbalance with advanced techniques
   - Trained and optimized 3 traditional ML algorithms
   - Implemented robust evaluation with TimeSeriesSplit
   - Crowned our Standard ML Champion

2. **🧠 Part 2 - Deep Learning Tournament**
   - Transformed data for sequential learning
   - Built and trained sophisticated neural networks
   - Discovered automatic temporal pattern recognition
   - Selected our Deep Learning Champion

3. **🏁 Part 3 - The Grand Finale**
   - Conducted ultimate head-to-head comparison
   - Declared our overall tournament champion
   - Prepared production-ready deployment artifacts
   - Created comprehensive documentation

### 🔑 Key Challenges Overcome

**Class Imbalance Crisis** 🚨
- **Challenge**: Severe imbalance in pest occurrence data
- **Solution**: Class weighting + optimal threshold tuning
- **Result**: Robust F1-score optimization for minority class

**Data Leakage Prevention** 🛡️
- **Challenge**: Temporal data requiring careful handling
- **Solution**: Chronological splits with shuffle=False
- **Result**: Realistic performance estimates for time-series

**Fair Model Comparison** ⚖️
- **Challenge**: Comparing different algorithm families
- **Solution**: Standardized evaluation framework
- **Result**: Objective champion selection process

**Feature Engineering vs Auto-Learning** 🤖
- **Challenge**: Traditional features vs neural discovery
- **Solution**: Separate tournaments with different data
- **Result**: Fair comparison of paradigms

### 📈 Scientific Achievements

✅ **Rigorous Methodology**: TimeSeriesSplit, class balancing, threshold optimization  
✅ **Comprehensive Evaluation**: F1, AUC, precision, recall, accuracy metrics  
✅ **Interactive Analysis**: 15+ Plotly visualizations for deep insights  
✅ **Reproducible Research**: All random seeds, saved results, detailed documentation  
✅ **Production Ready**: Deployment artifacts and comprehensive guides  



## 🏆 Final Thoughts

This tournament represents more than just model selection - it's a demonstration of:
- **Scientific rigor** in machine learning methodology
- **Practical application** of advanced data science techniques
- **Real-world problem solving** for agricultural challenges
- **Comprehensive evaluation** frameworks for production deployment

Our champion model is now ready to help farmers, agricultural consultants, and pest management professionals make data-driven decisions about pest occurrence, ultimately contributing to more efficient and sustainable agricultural practices.

**The tournament is complete. The champion is crowned. The future of pest prediction begins now!** 🌾🏆

---

*Thank you for joining us on this incredible machine learning journey!* 🎉